# Librerias

In [1]:
import hashlib
import json
import re
import sys
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
from pyproj import Geod, Transformer


def localizar_constantes(inicio=None):
    """Devuelve la carpeta que contiene constantes.py.

    Busca desde la ubicación del cuaderno hacia arriba. En cada nivel prueba
    la carpeta misma y su subcarpeta 'Bloque A', que es donde vive hoy el
    módulo.
    """
    inicio = Path(inicio or Path.cwd()).resolve()
    for base in [inicio, *inicio.parents]:
        for candidata in (base, base / "Bloque A"):
            if (candidata / "constantes.py").is_file():
                return candidata
    raise FileNotFoundError(
        "No se encontró constantes.py subiendo desde " + str(inicio)
    )


RUTA_CONSTANTES = localizar_constantes()
if str(RUTA_CONSTANTES) not in sys.path:
    sys.path.insert(0, str(RUTA_CONSTANTES))

import constantes

# Se imprime cuál se cargó: si hubiera una copia vieja del módulo en otra
# carpeta, el error aparece aquí y no diez celdas más abajo con números raros.
print("constantes.py cargado desde:", RUTA_CONSTANTES / "constantes.py")
print("RUTA_PROYECTO:            ", constantes.RUTA_PROYECTO)
print("Salidas de este cuaderno: ", constantes.ruta_entregable("B3"))

constantes.py cargado desde: F:\Tesis\Bloque A\constantes.py
RUTA_PROYECTO:             F:\Tesis
Salidas de este cuaderno:  F:\Tesis\Bloque B\Entregable B3


In [2]:
# =============================================================================
# BLOQUE B3 — Emparejamiento por criterios combinados
# Celda 2 — Rutas, parámetros y su procedencia
# =============================================================================
# ORDEN DEL BLOQUE B (lineal, sin ciclos):
#   B1    empareja por distancia; clasifica; deja 6 casos sin verificar
#   B2    mide la trama urbana y resuelve casos con manzana catastral (UAECD)
#   B2.1  aplica los veredictos y documenta el contraste contra B1
#   B3    reemplaza el emparejamiento por un puntaje combinado y lo contrasta
#
# B3 NO modifica las salidas de B1, B2 ni B2.1. Recalcula el emparejamiento
# desde las fuentes de A2 y A3 y produce sus propios archivos, de modo que el
# emparejamiento por distancia queda conservado como registro del método previo.
#
# Recalcular no significa quedar libre de lo ya verificado: B2 resolvió dos
# casos con una tercera fuente y B3 no puede deshacerlos. Esa restricción se
# aplica más abajo, en la celda de verificación externa.

RUTA_A2   = constantes.ruta_entregable("A2")
RUTA_A3   = constantes.ruta_entregable("A3")
RUTA_B1   = constantes.ruta_entregable("B1")
RUTA_B2   = constantes.ruta_entregable("B2")
RUTA_B2_1 = constantes.ruta_entregable("B2.1")
RUTA_OUT  = Path(constantes.ruta_entregable("B3"))
RUTA_OUT.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Sistemas de referencia
# -----------------------------------------------------------------------------
CRS_GEO  = constantes.CRS_WGS84     # EPSG:4326 — como vienen OSM y OCM
CRS_PROY = constantes.CRS_METRICO   # EPSG:9377 — MAGNA-SIRGAS, métrico

_tr = Transformer.from_crs(CRS_GEO, CRS_PROY, always_xy=True)

# -----------------------------------------------------------------------------
# Parámetros heredados
#
# Se importan de constantes.py y NO se redeclaran con su valor literal: dos
# copias del mismo umbral en dos bloques distintos divergen en cuanto alguien
# corrige una sola. La misma regla que ya sigue B1.
# -----------------------------------------------------------------------------
RADIO_BUSQUEDA_MAX = constantes.RADIO_BUSQUEDA_MAX  # 500 m — más allá no se postula pareja
RADIO_SECUNDARIO   = constantes.RADIO_SECUNDARIO    # 300 m — radio para vínculo N:1
RADIO_DUPLICADO    = constantes.RADIO_DUPLICADO     # 50 m  — duplicado interno
UMBRAL_TEXTUAL     = constantes.UMBRAL_TEXTUAL      # 0,45  — convención de similitud textual
UMBRAL_CONFIRMADA  = constantes.UMBRAL_CONFIRMADA   # 25 m  — error típico de GPS urbano
UMBRAL_PROBABLE    = constantes.UMBRAL_PROBABLE     # 100 m — lado largo mediano de manzana

# -----------------------------------------------------------------------------
# Escala de decaimiento espacial
#
# No es un número elegido: es la medida que B2 tomó sobre la capa catastral de
# la UAECD en el entorno de las estaciones. A esa distancia el componente
# espacial vale exp(-1) = 0,368, de modo que "una manzana de separación"
# equivale a perder dos tercios de la evidencia posicional.
#
# LIMITACIÓN DECLARADA: la medida es de Bogotá. No se buscó una capa de
# manzanas comparable para Medellín, de modo que la escala se transfiere allá
# sin verificar. Medellín aporta 2 de los 20 pares primarios y ninguno de los
# casos ambiguos, así que el alcance del supuesto es acotado, pero es un
# supuesto.
# -----------------------------------------------------------------------------
ESCALA_DISTANCIA_M = constantes.MEDIDA_ANCLAJE["lado_largo_mediano_m"]
ESCALA_CIUDAD_MEDIDA = constantes.MEDIDA_ANCLAJE["ciudad_medida"]
ESCALA_FUENTE = constantes.MEDIDA_ANCLAJE["fuente"]

# -----------------------------------------------------------------------------
# Pesos del puntaje combinado
#
# La distancia pesa más que el nombre porque es la única evidencia disponible
# en todos los pares: el nombre falta en una fracción de los registros y el
# operador en cerca de la mitad del inventario de OCM. Un criterio que dependa
# sobre todo de un campo ausente no clasifica, adivina.
#
# El operador entra como AJUSTE ADITIVO, nunca como componente principal, por
# la misma razón. Su aporte real se mide en la celda de sensibilidad: si no
# mueve ninguna asignación, debe declararse inactivo en lugar de figurar como
# criterio del método.
# -----------------------------------------------------------------------------
PESO_DISTANCIA = 0.60
PESO_NOMBRE    = 0.40

BONO_OPERADOR = 0.10   # ambos operadores resuelven al mismo alias canónico
VETO_OPERADOR = 0.00   # operadores distintos; se mide su efecto, no se supone

assert abs((PESO_DISTANCIA + PESO_NOMBRE) - 1.0) < 1e-9, (
    "Los pesos deben sumar 1: si no, el puntaje no es comparable contra un "
    "umbral fijo y cambiar un peso mueve la escala entera."
)

# -----------------------------------------------------------------------------
# Umbral de aceptación
#
# El umbral NO se calibra contra los veredictos de B2. Conviene decirlo con
# precisión porque una versión anterior de esta celda afirmaba lo contrario:
# B2 adoptó dos veredictos —C3-2 y C3-3—, los dos negativos, de modo que no
# existe un solo caso positivo contra el cual medir falsos negativos. Con dos
# etiquetas del mismo signo no se calibra nada; a lo sumo se comprueba que
# ningún umbral razonable reconstituya lo que la manzana descartó, y esa
# comprobación es una prueba de aceptación, no una calibración.
#
# El umbral se elige, entonces, con un criterio declarado y una tabla de
# sensibilidad:
#
#   Criterio: un falso positivo —dos estaciones distintas fusionadas en una—
#   es más grave que un falso negativo —una estación contada como dos
#   registros—, porque el inventario está destinado a depósito con
#   identificador persistente. Un registro de más se corrige después; una
#   fusión indebida borra una estación del inventario y nadie la echa de menos.
#
#   Evidencia: la tabla de la celda de sensibilidad muestra qué pares entran y
#   salen en cada escalón, de modo que la elección se lee con sus consecuencias
#   a la vista y no como un número dado.
#
# El valor adoptado se fija aquí para que el cuaderno corra de principio a fin,
# pero la celda de sensibilidad debe recorrerse antes de darlo por bueno.
# -----------------------------------------------------------------------------
UMBRALES_EVALUADOS = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

UMBRAL_PUNTAJE = 0.50

assert UMBRAL_PUNTAJE in UMBRALES_EVALUADOS, (
    "El umbral adoptado debe estar en la tabla de sensibilidad: si no, se "
    "estaría reportando un barrido que no incluye la decisión que se tomó."
)

# -----------------------------------------------------------------------------
print("PARÁMETROS DE B3")
print("-" * 78)
print(f"  Escala de distancia:  {ESCALA_DISTANCIA_M} m")
print(f"    medida en:          {ESCALA_CIUDAD_MEDIDA} · {ESCALA_FUENTE}")
print(f"  Pesos:                distancia {PESO_DISTANCIA} · nombre {PESO_NOMBRE}")
print(f"  Ajuste por operador:  bono {BONO_OPERADOR:+.2f} · veto {VETO_OPERADOR:+.2f}")
print(f"  Umbral adoptado:      {UMBRAL_PUNTAJE}")
print(f"  Umbrales evaluados:   {UMBRALES_EVALUADOS}")
print(f"  Radio de búsqueda:    {RADIO_BUSQUEDA_MAX} m")
print(f"\n  Salidas en: {RUTA_OUT}")

PARÁMETROS DE B3
------------------------------------------------------------------------------
  Escala de distancia:  101.7 m
    medida en:          Bogota · Manzana. Bogotá D.C. — UAECD / Mapa de Referencia IDECA
  Pesos:                distancia 0.6 · nombre 0.4
  Ajuste por operador:  bono +0.10 · veto +0.00
  Umbral adoptado:      0.5
  Umbrales evaluados:   [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]
  Radio de búsqueda:    500.0 m

  Salidas en: F:\Tesis\Bloque B\Entregable B3


# Procedencia de los insumos

In [3]:
ARCH_MANIFIESTO_B2 = constantes.exigir_archivo(
    constantes.ruta_entregable("B2", "B2_manifiesto_insumos.json"))

with open(ARCH_MANIFIESTO_B2, encoding="utf-8") as fh:
    MANIFIESTO_B2 = json.load(fh)

# --- Insumos que B3 consume --------------------------------------------------
# La clave es el NOMBRE DEL ARCHIVO, no una etiqueta propia, para que coincida
# con la del manifiesto de B2 y la comparación sea directa.
INSUMOS = {
    "A2_estaciones_OSM_Bogota.geojson":
        constantes.ruta_entregable("A2", "A2_estaciones_OSM_Bogota.geojson"),
    "A2_estaciones_OSM_Medellin.geojson":
        constantes.ruta_entregable("A2", "A2_estaciones_OSM_Medellin.geojson"),
    "A3_OpenChargeMap_Bogota_estaciones.csv":
        constantes.ruta_entregable("A3", "A3_OpenChargeMap_Bogota_estaciones.csv"),
    "A3_OpenChargeMap_Medellin_estaciones.csv":
        constantes.ruta_entregable("A3", "A3_OpenChargeMap_Medellin_estaciones.csv"),
    "B1_pares_primarios.csv":
        constantes.ruta_entregable("B1", "B1_pares_primarios.csv"),
    "B1_clasificacion_cuatro_categorias.csv":
        constantes.ruta_entregable("B1", "B1_clasificacion_cuatro_categorias.csv"),
    "B1_fusiones_internas.csv":
        constantes.ruta_entregable("B1", "B1_fusiones_internas.csv"),
    "B2_veredictos.json":
        constantes.ruta_entregable("B2", "B2_veredictos.json"),
    "B2_1_clasificacion_verificada.csv":
        constantes.ruta_entregable("B2.1", "B2_1_clasificacion_verificada.csv"),
}

HASHES_INSUMOS = {}
for nombre, ruta in INSUMOS.items():
    constantes.exigir_archivo(ruta)      # falla claro si no existe
    HASHES_INSUMOS[nombre] = constantes.sha256_archivo(ruta)

# --- Crudos congelados, heredados de B2 --------------------------------------
CRUDOS_HEREDADOS = {k: v for k, v in MANIFIESTO_B2.items()
                    if k.startswith("crudo::")}

if not CRUDOS_HEREDADOS:
    raise RuntimeError(
        "El manifiesto de B2 no trae entradas 'crudo::'. Sin ellas la cadena "
        "de procedencia queda cortada en los derivados y no llega a los "
        "archivos congelados de A2 y A3."
    )

# --- Comparación contra lo que consumió B2 -----------------------------------
# Solo se comparan los archivos que B2 también consumió. B3 usa además
# B1_fusiones_internas.csv y la salida de B2.1, que B2 no podía conocer: esos
# se registran sin comparar y se marcan como tales.
print("PROCEDENCIA DE LOS INSUMOS")
print("=" * 78)

discrepancias, nuevos = [], []
for nombre, calculado in HASHES_INSUMOS.items():
    registrado = MANIFIESTO_B2.get(nombre)
    if registrado is None:
        estado = "no comparable (posterior a B2)"
        nuevos.append(nombre)
    elif registrado == calculado:
        estado = "coincide con B2"
    else:
        estado = "NO COINCIDE con B2"
        discrepancias.append(nombre)
    print(f"  {nombre}")
    print(f"      {calculado[:16]}…  {estado}")

if discrepancias:
    raise RuntimeError(
        "Los insumos cambiaron respecto de los que consumió B2: "
        + ", ".join(discrepancias)
        + ".\nLos veredictos de manzana y la clasificación verificada "
        "describen otra versión de los datos, de modo que aplicarlos aquí "
        "asignaría una conclusión a una estación distinta de la que se "
        "verificó. Si la regeneración fue intencional, hay que correr la "
        "cadena completa hacia adelante: B1, B2, B2.1 y recién después B3."
    )

# --- Encadenamiento de B2.1 --------------------------------------------------
# B2.1 dejó su propio manifiesto. Si existe, se comprueba que B3 esté leyendo
# la misma versión de B1 que leyó B2.1: es el eslabón que conecta los dos
# cuadernos que consumen la clasificación.
ARCH_MANIFIESTO_B2_1 = constantes.ruta_entregable(
    "B2.1", "B2_1_manifiesto_insumos.json")

if Path(ARCH_MANIFIESTO_B2_1).exists():
    with open(ARCH_MANIFIESTO_B2_1, encoding="utf-8") as fh:
        MANIFIESTO_B2_1 = json.load(fh)
    _comunes = [k for k in HASHES_INSUMOS if k in MANIFIESTO_B2_1]
    _rotos = [k for k in _comunes if MANIFIESTO_B2_1[k] != HASHES_INSUMOS[k]]
    if _rotos:
        raise RuntimeError(
            "B3 y B2.1 están leyendo versiones distintas de: "
            + ", ".join(_rotos)
        )
    print(f"\n  Cadena con B2.1 verificada en {len(_comunes)} archivos comunes.")
else:
    print("\n  No se encontró B2_1_manifiesto_insumos.json; no se pudo "
          "encadenar con B2.1.")

print(f"\n  Insumos verificados:  {len(HASHES_INSUMOS) - len(nuevos)}")
print(f"  Registrados sin comparar (posteriores a B2): {len(nuevos)}")
for n in nuevos:
    print(f"      {n}")
print(f"  Crudos congelados heredados de B2: {len(CRUDOS_HEREDADOS)}")
for n in sorted(CRUDOS_HEREDADOS):
    print(f"      {n.removeprefix('crudo::')}")
print(f"\n  Fecha de corte del inventario: {constantes.FECHA_CORTE_LOCAL}")

PROCEDENCIA DE LOS INSUMOS
  A2_estaciones_OSM_Bogota.geojson
      99f132a15e2a87e9…  coincide con B2
  A2_estaciones_OSM_Medellin.geojson
      69a8c0bafbf86fce…  coincide con B2
  A3_OpenChargeMap_Bogota_estaciones.csv
      cb12e624d5aa1856…  coincide con B2
  A3_OpenChargeMap_Medellin_estaciones.csv
      02767749ab5e1857…  coincide con B2
  B1_pares_primarios.csv
      5709dadedd1c6f65…  coincide con B2
  B1_clasificacion_cuatro_categorias.csv
      4f6d8549d9d829f4…  coincide con B2
  B1_fusiones_internas.csv
      e8544de7dff500aa…  no comparable (posterior a B2)
  B2_veredictos.json
      b9e0d7ee9f101cb3…  no comparable (posterior a B2)
  B2_1_clasificacion_verificada.csv
      b0a2e926cf09b73c…  no comparable (posterior a B2)

  Cadena con B2.1 verificada en 3 archivos comunes.

  Insumos verificados:  6
  Registrados sin comparar (posteriores a B2): 3
      B1_fusiones_internas.csv
      B2_veredictos.json
      B2_1_clasificacion_verificada.csv
  Crudos congelados heredado

# Normalización y centinelas

In [4]:
# Dos cosas distintas viven en esta celda y conviene no confundirlas:
#
#   1. CENTINELAS — aporte propio de B3. OpenChargeMap escribe
#      «(Unknown Operator)» como texto literal cuando el operador no está
#      declarado. Tratarlo como nombre real contaminaba la similitud de
#      operador: dos estaciones sin operador declarado se «parecían» entre sí.
#
#   2. SIMILITUD TEXTUAL — NO es aporte de B3. Son exactamente las funciones
#      de B1, copiadas sin cambios. La razón es metodológica, no de comodidad:
#      B3 se presenta como un contraste contra el emparejamiento por distancia
#      de B1. Si mide el nombre con otra regla, deja de comparar métodos y pasa
#      a comparar dos pipelines, y las diferencias que reporte serían
#      atribuibles a la lista de palabras vacías tanto como al puntaje.
#
#      Una versión anterior de esta celda declaraba su propia lista de palabras
#      vacías, que incluía «parqueadero», «centro» y «comercial». Esas palabras
#      se sacaron de la lista compartida en su momento porque en estos nombres
#      sí discriminan —«C. C. Plaza Central», «Parqueadero universidad EAN»— y
#      reintroducirlas habría invalidado las medidas de A4 usadas como
#      justificación.
#
# La lista canónica vive en constantes.py y es común a A4, B1 y B3.

PALABRAS_NO_INFORMATIVAS = constantes.PALABRAS_NO_INFORMATIVAS

# -----------------------------------------------------------------------------
# Centinelas
# -----------------------------------------------------------------------------
# Valores que la fuente escribe para decir «no hay dato». Un centinela no es
# un nombre corto ni un nombre raro: es la ausencia del dato disfrazada de
# dato, y debe convertirse en None antes de cualquier comparación.
#
# Se comparan ya normalizados, de modo que basta declarar la forma sin tildes
# ni mayúsculas.

CENTINELAS = {
    "unknown operator",     # OpenChargeMap: literal '(Unknown Operator)'
    "unknown",
    "n a",                  # 'n/a' tras normalizar: la barra pasa a espacio
    "na",
    "none",
    "null",
    "sin nombre",
    "sin datos",
    "desconocido",
    "",
}


def normalizar(t):
    """Normaliza un texto para permitir comparaciones entre fuentes.

    Minúsculas, sin tildes, sin signos, con espacios colapsados. Todo lo que
    compara texto en este cuaderno pasa antes por aquí.
    """
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return ""
    s = str(t).lower()
    s = "".join(c for c in unicodedata.normalize("NFD", s)
                if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s)).strip()


def limpiar(t):
    """Devuelve el valor original, o None si es un centinela.

    Se devuelve el ORIGINAL y no el normalizado: la normalización sirve para
    comparar, no para almacenar. Lo que se escriba en las salidas debe seguir
    siendo lo que dice la fuente, para que cualquiera pueda verificarlo contra
    ella.
    """
    n = normalizar(t)
    return None if n in CENTINELAS else t


def obtener_tokens(valor):
    """Términos informativos de un texto, sin palabras de relleno."""
    texto = normalizar(valor)
    if not texto:
        return set()
    return {t for t in texto.split() if t not in PALABRAS_NO_INFORMATIVAS}


def similitud_tokens(a, b):
    """Jaccard entre los tokens informativos."""
    ta, tb = obtener_tokens(a), obtener_tokens(b)
    if not ta or not tb:
        return np.nan
    union = ta | tb
    if not union:
        return np.nan
    return len(ta & tb) / len(union)


def similitud_cadena(a, b):
    """Similitud de secuencia entre los textos normalizados."""
    na, nb = normalizar(a), normalizar(b)
    if not na or not nb:
        return np.nan
    return SequenceMatcher(None, na, nb).ratio()


def similitud(a, b):
    """Combina Jaccard y similitud de cadena; conserva el mayor.

    Devuelve NaN solo cuando ninguna de las dos vías pudo calcularse, es
    decir cuando falta el texto de alguno de los dos lados. Esa distinción
    importa: NaN significa «no comparable», no «no se parecen».

    Se conserva el max() y no se prioriza Jaccard porque en A4 se midió el
    efecto: de 190 parejas candidatas, 15 alcanzan el umbral solo por la vía
    de cadenas; en 13 de ellas la distancia decide igual, y las 2 restantes
    —Unicentro y EnelX Modelia, ambas bajo 25 m— son coincidencias genuinas
    que Jaccard pierde porque las fuentes escriben el nombre propio de otra
    forma.
    """
    valores = [v for v in (similitud_tokens(a, b), similitud_cadena(a, b))
               if not pd.isna(v)]
    return max(valores) if valores else np.nan


def contencion_token(a, b):
    """True si los tokens informativos de un nombre están dentro del otro.

    Complementa a similitud(): 'Compensar' contra 'Punto de carga Compensar
    Avenida 68' da 0,41 por cadena y 0,17 por Jaccard, de modo que el max()
    tampoco lo rescata, y aun así es el mismo sitio.

    ES UN BOOLEANO Y NO UN PUNTAJE, deliberadamente. Como fracción sobre el
    nombre más corto, la contención vale 1,00 en cuanto se comparte un solo
    token informativo: «EDS Terpel 68» contra «Terpel Las Vegas» daría 1,00
    por la sola marca comercial. Usada como evidencia adicional para fusionar
    registros de la MISMA fuente es útil; usada como componente del puntaje
    entre fuentes sería un generador de falsos positivos.
    """
    ta = {t for t in obtener_tokens(a) if len(t) > 2}
    tb = {t for t in obtener_tokens(b) if len(t) > 2}
    if not ta or not tb:
        return False
    return ta <= tb or tb <= ta


def proyectar(lon, lat):
    """Lleva coordenadas geográficas a EPSG:9377, en metros."""
    x, y = _tr.transform(np.asarray(lon, float), np.asarray(lat, float))
    return np.asarray(x), np.asarray(y)


# -----------------------------------------------------------------------------
# Comprobaciones de las funciones, no de los datos
#
# Anclan el comportamiento con casos reales del inventario. Si alguien cambia
# la lista de palabras vacías o el criterio de combinación, estas fallan aquí
# y no diez celdas más abajo con un conteo distinto y sin explicación.
# -----------------------------------------------------------------------------
assert normalizar("Estación de Carga Éxito Colina") == "estacion de carga exito colina"
assert normalizar(None) == "" and normalizar(np.nan) == ""

assert limpiar("(Unknown Operator)") is None, (
    "El centinela de OpenChargeMap debe anularse; si no, dos estaciones sin "
    "operador declarado se parecerían entre sí."
)
assert limpiar("Terpel") == "Terpel", "limpiar() debe devolver el valor original"

assert pd.isna(similitud("Compensar", None)), (
    "Sin texto de un lado la similitud es NaN —no comparable—, nunca 0."
)
assert contencion_token("Compensar", "Punto de carga Compensar Avenida 68"), (
    "La contención debe rescatar el caso que ni Jaccard ni cadena resuelven."
)

# Palabras que una versión anterior de B3 trataba como vacías y que sí
# discriminan: si volvieran a la lista, estos dos nombres se parecerían
# demasiado y la comprobación fallaría.
assert "parqueadero" not in PALABRAS_NO_INFORMATIVAS
assert "comercial" not in PALABRAS_NO_INFORMATIVAS

print("Normalización y similitud: funciones de B1, comprobaciones al día.")
print(f"  Palabras no informativas (constantes.py): {len(PALABRAS_NO_INFORMATIVAS)}")
print(f"  Centinelas declarados:                    {len(CENTINELAS)}")

Normalización y similitud: funciones de B1, comprobaciones al día.
  Palabras no informativas (constantes.py): 23
  Centinelas declarados:                    10


# Diccionario de alias de operador

In [5]:
# B1 comparaba operadores con la misma función de similitud textual que usaba
# para los nombres: «Enel X» contra «EVX» daba una similitud baja y el
# operador no aportaba evidencia, aunque fueran la misma red. B3 introduce una
# resolución por alias declarados.
#
# La consecuencia metodológica es que el contraste B1 -> B3 mezcla DOS cambios:
# el puntaje combinado y la normalización de operador. Si se reporta un solo
# número de diferencia, el lector no sabrá a cuál atribuirla. Por eso la celda
# de sensibilidad mide los dos por separado.
#
# CÓMO SE CONSTRUYÓ, Y POR QUÉ ASÍ:
#
# Cada equivalencia se declara con su justificación y su fuente externa. Un
# diccionario inferido de los propios datos —agrupando lo que aparece junto—
# sería circular: usaría el emparejamiento para justificar el criterio que
# produce el emparejamiento. Uno inventado por parecido de escritura sería
# peor, porque «Terpel» y «Voltex» no se parecen en nada y son lo mismo,
# mientras que dos operadores con nombres parecidos pueden ser distintos.
#
# UNA CONSECUENCIA QUE CONVIENE ANTICIPAR:
#
# El caso que en B1 quedó señalado para B3 —«Terpel Las Vegas» contra «Voltex
# Las Vegas», a 30,7 m, con 0,75 de similitud de cadena «pese a ser operadores
# distintos»— deja de ser un conflicto en cuanto este diccionario existe:
# ambos resuelven a TERPEL, que es la realidad comercial. El veto por operador
# distinto, pensado precisamente para ese caso, no puede activarse ahí. No es
# un defecto del veto: es que el diagnóstico de B1 partía de leer los nombres
# comerciales como si fueran empresas distintas. Queda documentado porque el
# caso está citado en el informe y hay que corregir su lectura.

ALIAS_OPERADOR = {
    # --- Enel X / Codensa / EVX ------------------------------------------
    # Enel X es la línea de negocio de Enel-Codensa y EVX es la aplicación
    # desde la que se gestionan sus estaciones. Los tres nombres designan la
    # misma red de carga.
    # Fuente: enelx.com/co/es/historias/nueva-estacion-de-carga-unicentro
    "enel x": "ENELX",
    "enelx": "ENELX",
    "enel": "ENELX",
    "enel codensa": "ENELX",
    "codensa": "ENELX",
    "evx": "ENELX",

    # --- Terpel / Voltex --------------------------------------------------
    # Voltex es la marca de electrolineras de Terpel. Las fuentes escriben
    # indistintamente la empresa, la marca o las dos juntas.
    "terpel": "TERPEL",
    "voltex": "TERPEL",
    "terpel voltex": "TERPEL",
    "voltex terpel": "TERPEL",
    "terpel voltex colombia": "TERPEL",
    "terpel voltex columbia": "TERPEL",   # erratum presente en la fuente

    # --- Sin equivalencias conocidas --------------------------------------
    # Se declaran igual, para dejar constancia de que se revisaron y no de
    # que se omitieron. Un operador ausente del diccionario no se resuelve, y
    # no resolverse no es lo mismo que ser distinto.
    "autogermana": "AUTOGERMANA",
    "mubon": "MUBON",
    "erco": "ERCO",
    "erco energia": "ERCO",
    "epm": "EPM",
    "celsia": "CELSIA",
    "universidad de los andes": "UNIANDES",
    "universiad de los andes": "UNIANDES",   # erratum presente en OSM
    "uniandes": "UNIANDES",
}

# Motivo de cada agrupación, para que el archivo de salida no dependa de leer
# los comentarios del cuaderno.
JUSTIFICACION_ALIAS = {
    "ENELX": ("Enel X es la línea de negocio de Enel-Codensa; EVX es su "
              "aplicación de gestión. Misma red."),
    "TERPEL": "Voltex es la marca de electrolineras de Terpel.",
    "AUTOGERMANA": "Sin equivalencias conocidas.",
    "MUBON": "Sin equivalencias conocidas.",
    "ERCO": "Sin equivalencias conocidas.",
    "EPM": "Sin equivalencias conocidas.",
    "CELSIA": "Sin equivalencias conocidas.",
    "UNIANDES": ("Mismo operador institucional; se recoge además un erratum "
                 "de escritura presente en OpenStreetMap."),
}


def id_operador(t):
    """Identificador canónico del operador, o None si no se puede resolver.

    TRES ESTADOS, NO DOS. Esta distinción es el punto entero de la función:

      - un identificador  -> el operador se conoce y se resolvió
      - None por centinela -> la fuente declara explícitamente que no hay dato
      - None por ausencia  -> el texto existe pero no está en el diccionario

    Los dos últimos se colapsan en None a propósito: en ambos casos el
    operador NO debe intervenir en el puntaje. Lo que nunca puede ocurrir es
    que un None se lea como «operador distinto» y penalice la pareja. No
    saber no es evidencia en contra.
    """
    t = limpiar(t)
    if t is None:
        return None
    return ALIAS_OPERADOR.get(normalizar(t))


def diagnosticar_operador(t):
    """Igual que id_operador, pero explica por qué falló. Para el reporte."""
    if limpiar(t) is None:
        return None, "sin dato (vacío o centinela)"
    clave = normalizar(t)
    if clave in ALIAS_OPERADOR:
        return ALIAS_OPERADOR[clave], "resuelto"
    return None, "no declarado en el diccionario"


# -----------------------------------------------------------------------------
# Comprobaciones del diccionario
# -----------------------------------------------------------------------------
# La búsqueda se hace sobre el texto normalizado, de modo que una clave
# escrita con mayúsculas, tildes o puntuación NUNCA coincidiría y el alias
# sería un no-op silencioso: el cuaderno correría igual, con el operador sin
# resolver y sin que nada lo advirtiera.
_mal_escritas = [k for k in ALIAS_OPERADOR if normalizar(k) != k]
assert not _mal_escritas, (
    f"Claves que no están normalizadas y nunca coincidirán: {_mal_escritas}. "
    "Escríbalas en minúscula, sin tildes y sin signos."
)

# Todo identificador canónico debe tener justificación escrita.
_sin_motivo = sorted(set(ALIAS_OPERADOR.values()) - set(JUSTIFICACION_ALIAS))
assert not _sin_motivo, (
    f"Identificadores sin justificación declarada: {_sin_motivo}. "
    "Una equivalencia sin motivo escrito no es verificable por un tercero."
)

# Casos que anclan el comportamiento.
assert id_operador("Enel X") == id_operador("EVX") == "ENELX"
assert id_operador("Terpel") == id_operador("Voltex") == "TERPEL", (
    "Terpel y Voltex deben resolver al mismo operador: es la razón por la "
    "que el veto no puede activarse en el caso señalado por B1."
)
assert id_operador("(Unknown Operator)") is None
assert id_operador("Operador Que No Existe") is None, (
    "Un operador no declarado devuelve None, no un identificador inventado."
)

# -----------------------------------------------------------------------------
_canonicos = sorted(set(ALIAS_OPERADOR.values()))
print("DICCIONARIO DE ALIAS DE OPERADOR")
print("-" * 78)
print(f"  Alias declarados:            {len(ALIAS_OPERADOR)}")
print(f"  Operadores canónicos:        {len(_canonicos)}")
print(f"  {', '.join(_canonicos)}")
print("\n  La cobertura real sobre los datos se mide en la celda de carga:")
print("  un diccionario completo en el papel puede resolver la mitad del")
print("  inventario si los operadores llegan escritos de otra forma.")

DICCIONARIO DE ALIAS DE OPERADOR
------------------------------------------------------------------------------
  Alias declarados:            21
  Operadores canónicos:        8
  AUTOGERMANA, CELSIA, ENELX, EPM, ERCO, MUBON, TERPEL, UNIANDES

  La cobertura real sobre los datos se mide en la celda de carga:
  un diccionario completo en el papel puede resolver la mitad del
  inventario si los operadores llegan escritos de otra forma.


# Carga y deduplicación

In [6]:
# La carga NO se reimplementa. constantes.cargar_fuentes_estaciones() lee los
# archivos congelados de A2 y A3, fija los identificadores como texto y
# resuelve el '475409.0' que ya rompió una verificación de la tutoría. Una
# segunda rutina de carga en B3 sería una segunda oportunidad de que las dos
# diverjan.
#
# OJO CON EL ORDEN: la función devuelve (osm, ocm), en ese orden. Invertirlo
# no produce ningún error —las dos tablas tienen las mismas columnas— y el
# cuaderno correría entero dando el emparejamiento al revés.
#
# La deduplicación tampoco se reimplementa: es la de B1, con cierre transitivo
# por union-find. Una versión anterior de B3 la reescribió con un conjunto de
# descartados y sin cerrar cadenas: si A se fusiona con B y B con C, el
# registro conservado podía ser uno que a su vez había sido absorbido. Hoy no
# hay cadenas de tres en estos datos, de modo que el resultado salía bien por
# suerte y no por construcción. Al final de la celda se comprueba contra las
# fusiones que registró B1.

osm, ocm = constantes.cargar_fuentes_estaciones()

print("CARGA DESDE LOS INSUMOS CONGELADOS")
print("-" * 78)
print(f"  OSM (A2): {len(osm):>3} registros   "
      f"(Bogotá {(osm.ciudad == 'Bogota').sum()}, "
      f"Medellín {(osm.ciudad == 'Medellin').sum()})")
print(f"  OCM (A3): {len(ocm):>3} registros   "
      f"(Bogotá {(ocm.ciudad == 'Bogota').sum()}, "
      f"Medellín {(ocm.ciudad == 'Medellin').sum()})")

# -----------------------------------------------------------------------------
# Anulación de centinelas
#
# Aporte propio de B3: B1 comparaba el texto tal como venía, de modo que dos
# estaciones con '(Unknown Operator)' se parecían entre sí. Aquí el centinela
# pasa a None ANTES de cualquier comparación.
#
# Se guarda el valor original en columnas aparte para que las salidas puedan
# verificarse contra la fuente sin tener que volver a A2 y A3.
# -----------------------------------------------------------------------------
for tabla in (osm, ocm):
    tabla["nombre_fuente"] = tabla["nombre"]
    tabla["operador_fuente"] = tabla["operador"]
    tabla["nombre"] = tabla["nombre"].map(limpiar)
    tabla["operador"] = tabla["operador"].map(limpiar)

_anulados = {
    "OSM nombre": int(osm.nombre.isna().sum() - osm.nombre_fuente.isna().sum()),
    "OSM operador": int(osm.operador.isna().sum() - osm.operador_fuente.isna().sum()),
    "OCM nombre": int(ocm.nombre.isna().sum() - ocm.nombre_fuente.isna().sum()),
    "OCM operador": int(ocm.operador.isna().sum() - ocm.operador_fuente.isna().sum()),
}
print("\n  Valores anulados por ser centinela, no vacío:")
for k, v in _anulados.items():
    print(f"    {k:<15} {v}")

# -----------------------------------------------------------------------------
# Proyección a metros
# -----------------------------------------------------------------------------
for tabla in (osm, ocm):
    tabla["x"], tabla["y"] = proyectar(tabla.lon, tabla.lat)

# Verificación independiente: la geometría debe reconstruir las coordenadas de
# origen al deshacer la proyección. Detecta un intercambio de argumentos en
# el transformador, que de otro modo desplazaría todo el inventario sin error.
_inv = Transformer.from_crs(CRS_PROY, CRS_GEO, always_xy=True)
for nombre_tabla, tabla in (("OSM", osm), ("OCM", ocm)):
    _lon, _lat = _inv.transform(tabla.x.values, tabla.y.values)
    assert np.allclose(_lon, tabla.lon.values, atol=1e-7), (
        f"{nombre_tabla}: la proyección no es reversible en longitud. "
        "Revise el orden de los argumentos en proyectar()."
    )
    assert np.allclose(_lat, tabla.lat.values, atol=1e-7), (
        f"{nombre_tabla}: la proyección no es reversible en latitud."
    )

# -----------------------------------------------------------------------------
# Cobertura real del diccionario de alias
#
# Un diccionario completo en el papel puede resolver la mitad del inventario
# si los operadores llegan escritos de otra forma. Esta tabla separa las dos
# causas de fallo, que se corrigen distinto: 'sin dato' es limitación de la
# fuente; 'no declarado' es tarea pendiente nuestra.
# -----------------------------------------------------------------------------
_diag = []
for nombre_tabla, tabla in (("OCM", ocm), ("OSM", osm)):
    for valor in tabla.operador_fuente:
        ident, motivo = diagnosticar_operador(valor)
        _diag.append({"fuente": nombre_tabla, "motivo": motivo,
                      "valor": valor, "identificador": ident})

COBERTURA_OPERADOR = pd.DataFrame(_diag)
_resumen_op = (COBERTURA_OPERADOR.groupby(["fuente", "motivo"]).size()
               .unstack(fill_value=0))

print("\nCOBERTURA DEL DICCIONARIO DE OPERADOR")
print("-" * 78)
print(_resumen_op.to_string())

_no_declarados = (COBERTURA_OPERADOR[COBERTURA_OPERADOR.motivo == "no declarado en el diccionario"]
                  .valor.dropna().value_counts())
if len(_no_declarados):
    print("\n  Valores presentes que el diccionario no resuelve:")
    for valor, n in _no_declarados.items():
        print(f"    {n:>3}x  {valor!r}")
    print("  No se agregan automáticamente: cada alias exige justificación")
    print("  externa declarada en la celda anterior.")
else:
    print("\n  Todos los operadores con dato resuelven a un identificador.")

# -----------------------------------------------------------------------------
# Deduplicación interna
#
# Colapsa registros de la MISMA fuente que describen el mismo punto. Es un
# problema distinto del emparejamiento entre fuentes y debe resolverse antes:
# un duplicado interno no tratado se presenta después como una estación sin
# contraparte e infla el vacío de cobertura, que es justamente lo que el
# trabajo quiere medir bien.
#
# Función tomada de B1 sin cambios de criterio.
# -----------------------------------------------------------------------------

def deduplicar(df, etiqueta):
    fusiones, descartar = [], set()

    for ciudad in df.ciudad.unique():
        S = df[df.ciudad == ciudad]
        idx = S.index.to_numpy()
        D = np.hypot(S.x.values[:, None] - S.x.values[None, :],
                     S.y.values[:, None] - S.y.values[None, :])
        np.fill_diagonal(D, np.inf)

        # --- 1. pares candidatos: cercanos y con evidencia textual ---
        vinculos = []
        for a, b in zip(*np.where(np.triu(D <= RADIO_DUPLICADO))):
            ia, ib = idx[a], idx[b]
            na, nb = df.at[ia, "nombre"], df.at[ib, "nombre"]
            s = similitud(na, nb)
            if (pd.notna(s) and s >= UMBRAL_TEXTUAL) or contencion_token(na, nb):
                vinculos.append((ia, ib, D[a, b]))

        if not vinculos:
            continue

        # --- 2. cierre transitivo: componentes conexas ---
        padre = {}

        def raiz(k):
            padre.setdefault(k, k)
            while padre[k] != k:
                padre[k] = padre[padre[k]]
                k = padre[k]
            return k

        for ia, ib, _ in vinculos:
            ra, rb = raiz(ia), raiz(ib)
            if ra != rb:
                padre[rb] = ra

        grupos = {}
        for k in padre:
            grupos.setdefault(raiz(k), []).append(k)

        # --- 3. un superviviente por grupo ---
        for miembros in grupos.values():
            conservado = min(miembros, key=lambda k: str(df.at[k, "id"]))
            for absorbido in miembros:
                if absorbido == conservado:
                    continue
                descartar.add(absorbido)
                d = next((v for ia, ib, v in vinculos
                          if {ia, ib} == {conservado, absorbido}), np.nan)
                fusiones.append({
                    "fuente": etiqueta, "ciudad": ciudad,
                    "id_conservado": df.at[conservado, "id"],
                    "id_absorbido": df.at[absorbido, "id"],
                    "nombre_conservado": df.at[conservado, "nombre"],
                    "nombre_absorbido": df.at[absorbido, "nombre"],
                    "miembros_del_grupo": len(miembros),
                    "distancia_m": round(d, 1) if pd.notna(d) else np.nan,
                })

    return df.drop(index=descartar).reset_index(drop=True), pd.DataFrame(fusiones)


def duplicados_no_detectables(df, etiqueta):
    """Pares cercanos que la regla no puede evaluar por falta de nombre.

    La fusión exige evidencia textual, de modo que dos registros sin nombre a
    pocos metros nunca se detectan. No se fusionan a ciegas —podrían ser
    puntos de carga distintos del mismo sitio, que es lo que el vínculo N:1
    modela—, pero su número acota cuántos duplicados internos pueden haber
    sobrevivido, y por eso se reporta.
    """
    casos = []
    for ciudad in df.ciudad.unique():
        S = df[df.ciudad == ciudad]
        idx = S.index.to_numpy()
        D = np.hypot(S.x.values[:, None] - S.x.values[None, :],
                     S.y.values[:, None] - S.y.values[None, :])
        np.fill_diagonal(D, np.inf)
        for a, b in zip(*np.where(np.triu(D <= RADIO_DUPLICADO))):
            ia, ib = idx[a], idx[b]
            na, nb = df.at[ia, "nombre"], df.at[ib, "nombre"]
            if pd.isna(similitud(na, nb)):
                casos.append({
                    "fuente": etiqueta, "ciudad": ciudad,
                    "id_a": df.at[ia, "id"], "id_b": df.at[ib, "id"],
                    "nombre_a": na, "nombre_b": nb,
                    "distancia_m": round(D[a, b], 1)})
    return pd.DataFrame(casos)


_osm_bruto_n, _ocm_bruto_n = len(osm), len(ocm)

osm, fusiones_osm = deduplicar(osm, "OSM")
ocm, fusiones_ocm = deduplicar(ocm, "OCM")

FUSIONES = pd.concat([fusiones_ocm, fusiones_osm], ignore_index=True)
NO_EVALUABLES = pd.concat(
    [duplicados_no_detectables(ocm, "OCM"),
     duplicados_no_detectables(osm, "OSM")], ignore_index=True)

print("\nDEDUPLICACIÓN INTERNA")
print("-" * 78)
print(f"  OCM: {_ocm_bruto_n} -> {len(ocm)}   ({len(fusiones_ocm)} fusiones)")
print(f"  OSM: {_osm_bruto_n} -> {len(osm)}   ({len(fusiones_osm)} fusiones)")
if len(FUSIONES):
    print()
    print(FUSIONES[["fuente", "ciudad", "id_conservado", "id_absorbido",
                    "nombre_conservado", "nombre_absorbido",
                    "miembros_del_grupo", "distancia_m"]].to_string(index=False))
print(f"\n  Pares cercanos sin nombre comparable, no evaluables: "
      f"{len(NO_EVALUABLES)}")
print("  (acotan cuántos duplicados internos pueden haber sobrevivido)")

# --- Verificación contra B1 --------------------------------------------------
# B1 registró sus fusiones internas. B3 recalcula desde las mismas fuentes con
# el mismo criterio, de modo que debe llegar exactamente a las mismas parejas.
# Se comparan por identificador, no por conteo: dos fusiones distintas también
# suman dos.
ARCH_FUSIONES_B1 = constantes.ruta_entregable("B1", "B1_fusiones_internas.csv")
if Path(ARCH_FUSIONES_B1).exists():
    _b1f = pd.read_csv(ARCH_FUSIONES_B1, encoding="utf-8-sig",
                       dtype={"id_conservado": "string",
                              "id_absorbido": "string"})
    _s_b1 = set(zip(_b1f.id_conservado.astype(str),
                    _b1f.id_absorbido.astype(str)))
    _s_b3 = set(zip(FUSIONES.id_conservado.astype(str),
                    FUSIONES.id_absorbido.astype(str))) if len(FUSIONES) else set()
    if _s_b1 == _s_b3:
        print(f"\n  Fusiones idénticas a las de B1: OK ({len(_s_b1)} parejas)")
    else:
        raise AssertionError(
            "La deduplicación de B3 no reproduce la de B1.\n"
            f"  Solo en B1: {sorted(_s_b1 - _s_b3)}\n"
            f"  Solo en B3: {sorted(_s_b3 - _s_b1)}\n"
            "B3 usa el mismo criterio y las mismas fuentes: una diferencia "
            "aquí significa que algo cambió en la similitud textual o en los "
            "insumos, no que B3 mejore la deduplicación."
        )
else:
    print("\n  No se encontró B1_fusiones_internas.csv; no se pudo contrastar.")

print(f"\nInventario deduplicado:  OCM {len(ocm)} | OSM {len(osm)}")

CARGA DESDE LOS INSUMOS CONGELADOS
------------------------------------------------------------------------------
  OSM (A2):  53 registros   (Bogotá 42, Medellín 11)
  OCM (A3):  38 registros   (Bogotá 27, Medellín 11)

  Valores anulados por ser centinela, no vacío:
    OSM nombre      0
    OSM operador    0
    OCM nombre      0
    OCM operador    13

COBERTURA DEL DICCIONARIO DE OPERADOR
------------------------------------------------------------------------------
motivo  no declarado en el diccionario  resuelto  sin dato (vacío o centinela)
fuente                                                                        
OCM                                  1        24                            13
OSM                                  4        20                            29

  Valores presentes que el diccionario no resuelve:
      1x  'Tesla (Tesla-only charging)'
      1x  'Hotel Tequendama'
      1x  'Car-B'
      1x  'Tesla'
      1x  'Isagen'
  No se agregan automáticamente

# Función de puntaje

In [7]:
# El puntaje es lo que reemplaza a la distancia como criterio de decisión. Su
# forma responde a tres restricciones que vienen de los datos, no de la teoría:
#
#   1. La distancia es la única evidencia disponible en TODOS los pares. El
#      nombre falta en una fracción de los registros y el operador en cerca de
#      la mitad del inventario de OpenChargeMap. Por eso pesa más y por eso es
#      la que queda sola cuando lo demás falta.
#
#   2. La ausencia de un atributo NO es evidencia en contra. Si a un par le
#      falta el nombre, el puntaje se reduce al componente espacial en lugar
#      de penalizarse. Penalizar la ausencia castigaría a la fuente peor
#      documentada, que es exactamente la que el trabajo quiere caracterizar.
#
#   3. El operador entra como AJUSTE ADITIVO, nunca como componente. Con la
#      mitad del campo vacío, hacerlo componente significaría que la mitad de
#      los pares se puntúa en una escala y la otra mitad en otra.
#
# DECAIMIENTO EXPONENCIAL, NO LINEAL NI ESCALONADO:
#
#   exp(-d/L) con L = 101,7 m (lado largo mediano de manzana, medido en B2).
#
#   Un corte escalonado —1 si está a menos de 25 m, 0 si no— reproduce el
#   método de B1 con otra notación y no aporta nada. Un decaimiento lineal
#   obligaría a declarar una distancia máxima arbitraria a partir de la cual
#   el aporte es cero. El exponencial no necesita ese corte: decae rápido
#   donde importa —a 25 m vale 0,78; a una manzana, 0,37; a 300 m, 0,05— y
#   nunca llega a cero, de modo que la distancia sigue ordenando candidatos
#   aun cuando todos estén lejos.
#
#   L no es un parámetro libre: es una medida externa. Cambiarlo exige volver
#   a medir la trama urbana, no elegir otro número.

def puntaje(distancia_m, nombre_a, nombre_b, operador_a, operador_b,
            peso_distancia=PESO_DISTANCIA, peso_nombre=PESO_NOMBRE,
            bono=BONO_OPERADOR, veto=VETO_OPERADOR,
            escala=ESCALA_DISTANCIA_M):
    """Puntaje combinado en [0,1] con el detalle de sus componentes.

    Los parámetros del método se reciben como argumentos y no se leen de
    variables globales. Es lo que permite que las celdas de sensibilidad
    barran valores sin mutar el estado del cuaderno: una versión anterior
    reasignaba la global con globals()[...] y la restauraba al final, de modo
    que una interrupción a mitad de celda dejaba el cuaderno corriendo con un
    parámetro distinto del declarado, sin aviso.

    Devuelve un diccionario y no un número porque cada decisión tiene que
    poder auditarse: con el puntaje solo, un par aceptado no se distingue de
    otro que llegó al mismo valor por un camino completamente distinto.
    """
    s_distancia = float(np.exp(-distancia_m / escala))
    s_nombre = similitud(nombre_a, nombre_b)

    # --- base: espacial, o espacial + textual si el nombre es comparable ---
    if pd.isna(s_nombre):
        base = s_distancia
        componentes = "distancia"
    else:
        base = peso_distancia * s_distancia + peso_nombre * s_nombre
        componentes = "distancia+nombre"

    # --- ajuste por operador ---
    # Los dos operadores tienen que RESOLVER para que el ajuste exista. Un
    # operador sin dato y otro sin dato no «coinciden»; un operador que el
    # diccionario no declara no «discrepa». En ambos casos el ajuste es cero
    # y queda registrado como tal, no como acuerdo ni como conflicto.
    id_a, id_b = id_operador(operador_a), id_operador(operador_b)

    if id_a is not None and id_b is not None:
        if id_a == id_b:
            ajuste = bono
            componentes += "+operador_igual"
        else:
            ajuste = veto
            componentes += "+operador_distinto"
    else:
        ajuste = 0.0
        componentes += "+operador_no_resoluble"

    total = float(np.clip(base + ajuste, 0.0, 1.0))

    return {
        "puntaje": total,
        "puntaje_sin_ajuste": float(np.clip(base, 0.0, 1.0)),
        "s_distancia": s_distancia,
        "s_nombre": s_nombre,
        "ajuste_operador": float(ajuste),
        "componentes": componentes,
        "op_ocm": id_a,
        "op_osm": id_b,
    }


# -----------------------------------------------------------------------------
# Comprobaciones de la función
#
# Anclan el comportamiento en los tres puntos donde una implementación
# razonable puede equivocarse sin que nada falle.
# -----------------------------------------------------------------------------

# 1. La escala hace lo que dice el comentario.
_a_una_manzana = puntaje(ESCALA_DISTANCIA_M, None, None, None, None)["s_distancia"]
assert abs(_a_una_manzana - np.exp(-1)) < 1e-12, (
    "A la distancia de escala el componente espacial debe valer exp(-1)."
)

# 2. La ausencia de nombre reduce al componente espacial, no penaliza.
_sin_nombre = puntaje(30.0, None, None, None, None)
_con_nombre_nulo = puntaje(30.0, "Compensar", None, None, None)
assert _sin_nombre["puntaje"] == _con_nombre_nulo["puntaje"], (
    "Falte el nombre de un lado o de los dos, el resultado debe ser el mismo: "
    "en ambos casos no es comparable."
)
assert _sin_nombre["componentes"].startswith("distancia+operador"), (
    "Sin nombre comparable, el componente textual no debe figurar."
)
assert _sin_nombre["puntaje"] > puntaje(30.0, "Compensar", "Éxito Colina",
                                        None, None)["puntaje"], (
    "Un par sin nombre comparable no puede puntuar MENOS que uno con nombres "
    "que no se parecen: eso sería penalizar la ausencia de dato."
)

# 3. Los tres estados del operador se distinguen.
_igual = puntaje(50.0, "EDS Terpel", "Estación Voltex", "Terpel", "Voltex")
assert _igual["ajuste_operador"] == BONO_OPERADOR
assert _igual["op_ocm"] == _igual["op_osm"] == "TERPEL", (
    "Terpel y Voltex son la misma red: el caso señalado por B1 como "
    "«operadores distintos» no puede activar el veto."
)

_sin_op = puntaje(50.0, "EDS Terpel", "Estación Voltex", "(Unknown Operator)", None)
assert _sin_op["ajuste_operador"] == 0.0
assert _sin_op["componentes"].endswith("operador_no_resoluble"), (
    "No resolver el operador no es lo mismo que discrepar."
)

_distinto = puntaje(50.0, "Estación", "Estación", "Terpel", "ERCO")
assert _distinto["componentes"].endswith("operador_distinto")
assert _distinto["ajuste_operador"] == VETO_OPERADOR

# 4. El puntaje queda acotado y el ajuste no puede sacarlo del rango.
for _d in (0.0, 1.0, 25.0, 100.0, 500.0, 5000.0):
    _p = puntaje(_d, "Unicentro", "Unicentro", "Terpel", "Voltex")
    assert 0.0 <= _p["puntaje"] <= 1.0

# -----------------------------------------------------------------------------
print("FUNCIÓN DE PUNTAJE")
print("-" * 78)
print(f"  base = {PESO_DISTANCIA} · exp(-d/{ESCALA_DISTANCIA_M}) "
      f"+ {PESO_NOMBRE} · similitud_nombre")
print(f"  ajuste = {BONO_OPERADOR:+.2f} si el operador coincide · "
      f"{VETO_OPERADOR:+.2f} si discrepa · 0 si no resuelve")
print("\n  Componente espacial por distancia:")
for _d in (0, 10, 25, 50, 100, 150, 300, 500):
    print(f"    {_d:>4} m  ->  {np.exp(-_d / ESCALA_DISTANCIA_M):.3f}")
print("\n  Comprobaciones de la función: OK")

FUNCIÓN DE PUNTAJE
------------------------------------------------------------------------------
  base = 0.6 · exp(-d/101.7) + 0.4 · similitud_nombre
  ajuste = +0.10 si el operador coincide · +0.00 si discrepa · 0 si no resuelve

  Componente espacial por distancia:
       0 m  ->  1.000
      10 m  ->  0.906
      25 m  ->  0.782
      50 m  ->  0.612
     100 m  ->  0.374
     150 m  ->  0.229
     300 m  ->  0.052
     500 m  ->  0.007

  Comprobaciones de la función: OK


# Emparejamiento por puntaje

In [8]:
# CAMBIO DE MÉTODO RESPECTO DE B1, Y POR QUÉ:
#
# B1 empareja por reciprocidad de distancia: se acepta la pareja si cada
# registro es el más cercano del otro. Es simétrico por construcción y no
# necesita llevar cuentas de qué está tomado. Su límite es que la distancia
# decide sola, y eso produjo el caso que el propio B1 dejó señalado: la
# estación «EnelX San Andresito» quedó emparejada con «Punto de carga Parque
# Los Ejidos» a 82,5 m con similitud de nombre 0,12, mientras «Punto de carga
# San Andresito Carrera 38» estaba a 120,2 m con similitud 0,52. La más
# cercana no era la correcta.
#
# B3 puntúa TODOS los candidatos dentro del radio y asigna 1:1 en orden
# descendente de puntaje. La asignación codiciosa no es simétrica por
# construcción como la recíproca, de modo que al final de la celda se compara
# contra el emparejamiento mutuo y se reporta la diferencia en lugar de
# suponerla nula.
#
# DESEMPATE DETERMINISTA:
#
# Dos candidatos con el mismo puntaje se ordenan por distancia y, si también
# empatan, por identificador. Sin ese último criterio el resultado dependería
# del orden de las filas en el archivo de entrada, y dos corridas sobre los
# mismos datos podrían diferir. El determinismo no es una comodidad: las
# salidas se comparan por hash entre corridas.
#
# MOTIVOS DE RECHAZO SEPARADOS:
#
# Un candidato puede descartarse por tres razones distintas, que llevan a
# conclusiones opuestas y no deben mezclarse:
#   - puntaje insuficiente          -> no son la misma estación
#   - el registro OCM ya fue tomado -> puede ser un punto adicional del mismo
#                                      sitio (vínculo N:1, celda siguiente)
#   - el registro OSM ya fue tomado -> ídem
# Colapsarlos en «registro ya emparejado» perdería los N:1, que es justo lo
# que B1 modela con la categoría C5.


def emparejar(ocm_df, osm_df, umbral=UMBRAL_PUNTAJE, radio=RADIO_BUSQUEDA_MAX,
              peso_distancia=PESO_DISTANCIA, peso_nombre=PESO_NOMBRE,
              bono=BONO_OPERADOR, veto=VETO_OPERADOR,
              escala=ESCALA_DISTANCIA_M, registrar_detalle=True):
    """Asignación 1:1 codiciosa por puntaje descendente.

    Devuelve (pares, rechazados, candidatos). 'candidatos' trae TODOS los
    pares evaluados con su puntaje, aceptados o no: es la materia prima de
    las celdas de sensibilidad, que así no tienen que reevaluar nada.

    Todos los parámetros del método son argumentos. Ninguna celda posterior
    debe mutar una global para barrer valores.
    """
    pares, rechazados, candidatos = [], [], []

    for ciudad in sorted(set(ocm_df.ciudad) | set(osm_df.ciudad)):
        # Índices LOCALES por ciudad: se reinician con reset_index, de modo
        # que mezclarlos entre ciudades uniría filas equivocadas.
        A = ocm_df[ocm_df.ciudad == ciudad].reset_index(drop=True)
        B = osm_df[osm_df.ciudad == ciudad].reset_index(drop=True)
        if A.empty or B.empty:
            continue

        D = np.hypot(A.x.values[:, None] - B.x.values[None, :],
                     A.y.values[:, None] - B.y.values[None, :])

        evaluados = []
        for i in range(len(A)):
            for j in range(len(B)):
                d = D[i, j]
                if d > radio:
                    continue
                p = puntaje(d, A.at[i, "nombre"], B.at[j, "nombre"],
                            A.at[i, "operador"], B.at[j, "operador"],
                            peso_distancia=peso_distancia,
                            peso_nombre=peso_nombre,
                            bono=bono, veto=veto, escala=escala)
                fila = {
                    "ciudad": ciudad,
                    "ocm_id": A.at[i, "id"], "osm_uid": B.at[j, "id"],
                    "nombre_ocm": A.at[i, "nombre"], "nombre_osm": B.at[j, "nombre"],
                    "operador_ocm": A.at[i, "operador"],
                    "operador_osm": B.at[j, "operador"],
                    "distancia_m": float(d),
                    **p,
                }
                evaluados.append((i, j, fila))
                if registrar_detalle:
                    candidatos.append(fila)

        # Orden determinista: puntaje desc, distancia asc, identificadores.
        evaluados.sort(key=lambda t: (-t[2]["puntaje"], t[2]["distancia_m"],
                                      str(t[2]["ocm_id"]), str(t[2]["osm_uid"])))

        tomados_ocm, tomados_osm = set(), set()
        for i, j, fila in evaluados:
            if fila["puntaje"] < umbral:
                if registrar_detalle:
                    rechazados.append({**fila,
                                       "motivo_rechazo": "puntaje insuficiente"})
                continue
            if i in tomados_ocm:
                if registrar_detalle:
                    rechazados.append({**fila,
                                       "motivo_rechazo": "OCM ya emparejada"})
                continue
            if j in tomados_osm:
                if registrar_detalle:
                    rechazados.append({**fila,
                                       "motivo_rechazo": "OSM ya emparejado"})
                continue
            tomados_ocm.add(i)
            tomados_osm.add(j)
            pares.append(fila)

    P = pd.DataFrame(pares)
    if len(P):
        P = (P.sort_values(["ciudad", "puntaje", "distancia_m"],
                           ascending=[True, False, True])
             .reset_index(drop=True))
        for col in ("ocm_id", "osm_uid"):
            P[col] = P[col].astype("string")

    R = pd.DataFrame(rechazados)
    C = pd.DataFrame(candidatos)
    return P, R, C


def emparejar_mutuo(ocm_df, osm_df, umbral=UMBRAL_PUNTAJE,
                    radio=RADIO_BUSQUEDA_MAX, **kwargs):
    """Variante recíproca: se acepta el par si cada registro es el MEJOR
    puntaje del otro. No es el método adoptado; sirve como verificación
    independiente del codicioso, por un camino distinto.
    """
    _, _, C = emparejar(ocm_df, osm_df, umbral=0.0, radio=radio,
                        registrar_detalle=True, **kwargs)
    if not len(C):
        return pd.DataFrame()

    mejor_ocm = (C.sort_values(["puntaje", "distancia_m"],
                               ascending=[False, True])
                 .groupby("ocm_id").first()["osm_uid"].to_dict())
    mejor_osm = (C.sort_values(["puntaje", "distancia_m"],
                               ascending=[False, True])
                 .groupby("osm_uid").first()["ocm_id"].to_dict())

    filas = [f for _, f in C.iterrows()
             if mejor_ocm.get(f.ocm_id) == f.osm_uid
             and mejor_osm.get(f.osm_uid) == f.ocm_id
             and f.puntaje >= umbral]
    return pd.DataFrame(filas).reset_index(drop=True)


PARES_B3, RECHAZADOS, CANDIDATOS = emparejar(ocm, osm)

print("EMPAREJAMIENTO POR PUNTAJE COMBINADO")
print("-" * 78)
print(f"  Candidatos evaluados (dentro de {RADIO_BUSQUEDA_MAX:.0f} m): {len(CANDIDATOS)}")
print(f"  Pares aceptados (puntaje >= {UMBRAL_PUNTAJE}):              {len(PARES_B3)}")
print(f"  Candidatos rechazados:                                 {len(RECHAZADOS)}")
if len(RECHAZADOS):
    print("\n  Por motivo:")
    for motivo, n in RECHAZADOS.motivo_rechazo.value_counts().items():
        print(f"    {n:>4}  {motivo}")
    print("\n  'OCM/OSM ya emparejado' NO significa que no sean el mismo sitio:")
    print("  puede ser un punto de carga adicional. Se resuelve en la celda")
    print("  de vínculos secundarios, no aquí.")

# -----------------------------------------------------------------------------
# Verificación independiente: codicioso frente a recíproco
#
# Son dos algoritmos distintos sobre el mismo puntaje. Si coinciden, la
# asignación no depende del orden de resolución. Si difieren, la diferencia
# es exactamente el conjunto de casos donde un registro fue disputado por dos
# candidatos, y hay que mirarlos uno por uno.
# -----------------------------------------------------------------------------
MUTUOS = emparejar_mutuo(ocm, osm)

_s_cod = set(zip(PARES_B3.ocm_id, PARES_B3.osm_uid)) if len(PARES_B3) else set()
_s_mut = set(zip(MUTUOS.ocm_id, MUTUOS.osm_uid)) if len(MUTUOS) else set()

print("\nCODICIOSO FRENTE A RECÍPROCO")
print("-" * 78)
print(f"  Codicioso: {len(_s_cod)}   Recíproco: {len(_s_mut)}   "
      f"Comunes: {len(_s_cod & _s_mut)}")
if _s_cod == _s_mut:
    print("  Coinciden: la asignación no depende del orden de resolución.")
else:
    print("  DIFIEREN. Casos disputados, para revisar uno por uno:")
    for oid, ouid in sorted(_s_cod - _s_mut):
        print(f"    solo codicioso: {oid} <-> {ouid}")
    for oid, ouid in sorted(_s_mut - _s_cod):
        print(f"    solo recíproco: {oid} <-> {ouid}")

# -----------------------------------------------------------------------------
# Reasignaciones: dónde el puntaje eligió distinto que la distancia sola
#
# Es el resultado que justifica el bloque entero. Para cada OCM emparejada, se
# compara el registro elegido por puntaje contra el que habría elegido la
# distancia mínima. Donde no coinciden, el criterio combinado cambió la
# decisión, y esos son los casos que hay que poder defender uno por uno.
# -----------------------------------------------------------------------------
REASIGNACIONES = pd.DataFrame()
if len(CANDIDATOS):
    _mas_cercano = (CANDIDATOS.sort_values("distancia_m")
                    .groupby("ocm_id").first())
    filas = []
    for _, f in PARES_B3.iterrows():
        if f.ocm_id not in _mas_cercano.index:
            continue
        cerc = _mas_cercano.loc[f.ocm_id]
        if cerc.osm_uid != f.osm_uid:
            filas.append({
                "ciudad": f.ciudad, "ocm_id": f.ocm_id,
                "nombre_ocm": f.nombre_ocm,
                "elegido_por_puntaje": f.osm_uid,
                "nombre_elegido": f.nombre_osm,
                "d_elegido": round(f.distancia_m, 1),
                "sim_elegido": f.s_nombre,
                "puntaje_elegido": round(f.puntaje, 3),
                "mas_cercano": cerc.osm_uid,
                "nombre_mas_cercano": cerc.nombre_osm,
                "d_mas_cercano": round(cerc.distancia_m, 1),
                "sim_mas_cercano": cerc.s_nombre,
                "puntaje_mas_cercano": round(cerc.puntaje, 3),
            })
    REASIGNACIONES = pd.DataFrame(filas)

print("\nREASIGNACIONES RESPECTO DE LA DISTANCIA MÍNIMA")
print("-" * 78)
print(f"  Pares donde el puntaje eligió un registro distinto del más "
      f"cercano: {len(REASIGNACIONES)}")
if len(REASIGNACIONES):
    print()
    print(REASIGNACIONES[["ocm_id", "nombre_ocm", "nombre_elegido",
                          "d_elegido", "sim_elegido", "puntaje_elegido",
                          "nombre_mas_cercano", "d_mas_cercano",
                          "sim_mas_cercano"]].round(3).to_string(index=False))
    print("\n  Cada una de estas filas debe poder defenderse en sustentación.")
    print("  Si alguna no se puede, el problema no es el umbral: es el peso")
    print("  del componente textual.")
else:
    print("  Ninguna. El puntaje ordena distinto pero elige lo mismo que la")
    print("  distancia, lo cual es en sí un resultado: significa que en este")
    print("  inventario el nombre no aporta información que la posición no dé.")

if len(PARES_B3):
    print("\nPARES ACEPTADOS")
    print("-" * 78)
    print(PARES_B3[["ciudad", "ocm_id", "nombre_ocm", "nombre_osm",
                    "distancia_m", "s_distancia", "s_nombre",
                    "ajuste_operador", "puntaje", "componentes"]]
          .round(3).to_string(index=False))

EMPAREJAMIENTO POR PUNTAJE COMBINADO
------------------------------------------------------------------------------
  Candidatos evaluados (dentro de 500 m): 27
  Pares aceptados (puntaje >= 0.5):              13
  Candidatos rechazados:                                 14

  Por motivo:
      14  puntaje insuficiente

  'OCM/OSM ya emparejado' NO significa que no sean el mismo sitio:
  puede ser un punto de carga adicional. Se resuelve en la celda
  de vínculos secundarios, no aquí.

CODICIOSO FRENTE A RECÍPROCO
------------------------------------------------------------------------------
  Codicioso: 13   Recíproco: 13   Comunes: 13
  Coinciden: la asignación no depende del orden de resolución.

REASIGNACIONES RESPECTO DE LA DISTANCIA MÍNIMA
------------------------------------------------------------------------------
  Pares donde el puntaje eligió un registro distinto del más cercano: 0
  Ninguna. El puntaje ordena distinto pero elige lo mismo que la
  distancia, lo cual es en sí 

# Contraste con el emparejamiento de B1

In [9]:
#   B1   — 20 pares primarios, de los cuales 17 se sostienen y 3 quedaron
#          DEGRADADOS (parejas lejanas sin respaldo textual: no eran pares).
#          Contrastar contra estos 17 mide el EFECTO DEL MÉTODO: qué cambia
#          al reemplazar la distancia por el puntaje combinado.
#
#   B2.1 — la clasificación verificada. B2 resolvió dos casos con la manzana
#          catastral de la UAECD y B2.1 disolvió los pares correspondientes,
#          de modo que la referencia vigente son 15 pares. Contrastar contra
#          estos mide el EFECTO NETO sobre el inventario que hoy es canónico.
#
# Mezclarlos daría un solo número de diferencia imposible de interpretar: no
# se sabría cuánto viene del puntaje y cuánto de una verificación externa que
# B3 no produjo.
#
# ADEMÁS, EL CONTRASTE CONTRA B1 MEZCLA DOS CAMBIOS: el puntaje combinado y la
# normalización de operador por alias, que B1 no tenía. Se separan al final de
# la celda reejecutando el emparejamiento con el ajuste de operador apagado.

# -----------------------------------------------------------------------------
# Carga de las dos líneas base
# -----------------------------------------------------------------------------
ARCH_PARES_B1 = constantes.exigir_archivo(
    constantes.ruta_entregable("B1", "B1_pares_primarios.csv"))
ARCH_VERIFICADA = constantes.exigir_archivo(
    constantes.ruta_entregable("B2.1", "B2_1_clasificacion_verificada.csv"))

# Los identificadores se leen como texto. Con nulos presentes pandas los
# infiere float y escribe '475140.0', que no une contra '475140': ninguna
# comparación cruzaría y el contraste saldría vacío sin error.
B1P = pd.read_csv(ARCH_PARES_B1, encoding="utf-8-sig",
                  dtype={"ocm_id": "string", "osm_uid": "string"})
VERIF = pd.read_csv(ARCH_VERIFICADA, encoding="utf-8-sig",
                    dtype={"ocm_id": "string", "osm_uid": "string"})
for tabla in (B1P, VERIF):
    for col in ("ocm_id", "osm_uid"):
        tabla[col] = tabla[col].str.replace(r"\.0$", "", regex=True)

# B1: los degradados no eran parejas válidas. Se excluyen de la línea base y
# se cuentan aparte, porque B3 tiene que poder mostrar que tampoco los acepta.
B1_DEGRADADOS = B1P[B1P.categoria == "DEGRADADO"].copy()
B1_VALIDOS = B1P[B1P.categoria != "DEGRADADO"].copy()

# B2.1: los pares vigentes son las filas con origen AMBAS, es decir aquellas
# donde las dos fuentes describen la misma estación después de la verificación.
VERIF_PARES = VERIF[(VERIF.origen == "AMBAS")
                    & VERIF.ocm_id.notna() & VERIF.osm_uid.notna()].copy()

# Los casos que B2 descartó con evidencia externa. B3 no puede reconstituirlos.
DESCARTADOS_B2 = VERIF[(VERIF.verificacion_externa == "descartado")
                       & VERIF.caso.notna()]
PARES_PROHIBIDOS = set()
for caso in sorted(DESCARTADOS_B2.caso.unique()):
    filas = DESCARTADOS_B2[DESCARTADOS_B2.caso == caso]
    _ocm = filas.ocm_id.dropna().unique()
    _osm = filas.osm_uid.dropna().unique()
    for o in _ocm:
        for s in _osm:
            PARES_PROHIBIDOS.add((o, s))

s_b1 = set(zip(B1_VALIDOS.ocm_id, B1_VALIDOS.osm_uid))
s_b21 = set(zip(VERIF_PARES.ocm_id, VERIF_PARES.osm_uid))
s_b3 = set(zip(PARES_B3.ocm_id, PARES_B3.osm_uid)) if len(PARES_B3) else set()

print("=" * 78)
print("LÍNEAS BASE")
print("=" * 78)
print(f"  B1  pares primarios válidos:        {len(s_b1)}"
      f"   (+{len(B1_DEGRADADOS)} degradados, no eran pares)")
print(f"  B2.1 pares tras verificación:       {len(s_b21)}")
print(f"  B3  pares por puntaje combinado:    {len(s_b3)}")
print(f"\n  Pares que B2 descartó con evidencia externa: {len(PARES_PROHIBIDOS)}")
for o, s in sorted(PARES_PROHIBIDOS):
    print(f"    {o} <-> {s}")


def detallar(conjunto, etiqueta):
    """Imprime los pares de un conjunto con la información disponible."""
    for oid, ouid in sorted(conjunto):
        en_b3 = PARES_B3[(PARES_B3.ocm_id == oid) & (PARES_B3.osm_uid == ouid)]
        en_b1 = B1P[(B1P.ocm_id == oid) & (B1P.osm_uid == ouid)]
        if len(en_b3):
            f = en_b3.iloc[0]
            print(f"    {oid} <-> {ouid}")
            print(f"        {f.nombre_ocm}  ||  {f.nombre_osm}")
            print(f"        {f.distancia_m:.1f} m · nombre "
                  f"{f.s_nombre if pd.notna(f.s_nombre) else float('nan'):.2f} · "
                  f"puntaje {f.puntaje:.3f}")
        elif len(en_b1):
            f = en_b1.iloc[0]
            sim = f.similitud_nombre
            print(f"    {oid} <-> {ouid}")
            print(f"        {f.nombre_ocm}  ||  {f.nombre_osm}")
            print(f"        {f.distancia_m:.1f} m · nombre "
                  f"{sim if pd.notna(sim) else float('nan'):.2f} · "
                  f"categoría B1: {f.categoria}")
            # Por qué B3 no lo aceptó
            rech = RECHAZADOS[(RECHAZADOS.ocm_id == oid)
                              & (RECHAZADOS.osm_uid == ouid)] \
                if len(RECHAZADOS) else pd.DataFrame()
            if len(rech):
                r = rech.iloc[0]
                print(f"        B3 lo rechazó: {r.motivo_rechazo} "
                      f"(puntaje {r.puntaje:.3f})")
            else:
                print(f"        B3 no lo evaluó: fuera del radio de "
                      f"{RADIO_BUSQUEDA_MAX:.0f} m")
        else:
            print(f"    {oid} <-> {ouid}   (sin detalle disponible)")


# -----------------------------------------------------------------------------
# Contraste 1 — efecto del método
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("CONTRASTE 1 · B1 (distancia recíproca)  ->  B3 (puntaje combinado)")
print("=" * 78)
print(f"  En B1: {len(s_b1)}   En B3: {len(s_b3)}   Comunes: {len(s_b1 & s_b3)}")

print(f"\n--- Pares de B1 que B3 NO sostiene ({len(s_b1 - s_b3)}) ---")
detallar(s_b1 - s_b3, "solo B1")

print(f"\n--- Pares que B3 añade y B1 no tenía ({len(s_b3 - s_b1)}) ---")
detallar(s_b3 - s_b1, "solo B3")

# Los degradados de B1 son un control: eran parejas que la distancia postuló y
# el criterio textual rechazó. Si B3 acepta alguno, está siendo más permisivo
# que B1 justo donde B1 ya había decidido no afirmar nada.
_deg = set(zip(B1_DEGRADADOS.ocm_id, B1_DEGRADADOS.osm_uid))
_deg_aceptados = _deg & s_b3
print(f"\n--- Control: degradados de B1 que B3 acepta ({len(_deg_aceptados)} "
      f"de {len(_deg)}) ---")
if _deg_aceptados:
    detallar(_deg_aceptados, "degradado aceptado")
    print("    Revisar: B1 los descartó por ser parejas lejanas sin respaldo")
    print("    textual. Aceptarlos exige justificar qué evidencia nueva aporta")
    print("    el puntaje que la regla anterior no veía.")
else:
    print("    Ninguno. B3 no reabre parejas que B1 había descartado.")

# -----------------------------------------------------------------------------
# Contraste 2 — efecto neto sobre el inventario canónico
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("CONTRASTE 2 · B2.1 (verificada)  ->  B3 (puntaje combinado)")
print("=" * 78)
print(f"  En B2.1: {len(s_b21)}   En B3: {len(s_b3)}   "
      f"Comunes: {len(s_b21 & s_b3)}")

print(f"\n--- Pares verificados que B3 NO sostiene ({len(s_b21 - s_b3)}) ---")
detallar(s_b21 - s_b3, "solo B2.1")

print(f"\n--- Pares que B3 añade sobre la clasificación verificada "
      f"({len(s_b3 - s_b21)}) ---")
detallar(s_b3 - s_b21, "solo B3")

# --- Contradicciones con la verificación externa ----------------------------
# Se REPORTAN aquí y se convierten en prueba dura en la celda de aceptación.
# Un par que B2 descartó mirando la manzana catastral no puede volver a
# afirmarse desde un puntaje calculado sobre las mismas dos fuentes que ya
# estaban en desacuerdo.
CONTRADICCIONES = sorted(s_b3 & PARES_PROHIBIDOS)
print(f"\n--- Contradicciones con la verificación externa de B2 "
      f"({len(CONTRADICCIONES)}) ---")
if CONTRADICCIONES:
    for oid, ouid in CONTRADICCIONES:
        print(f"    {oid} <-> {ouid}  — B2 los declaró estaciones distintas")
    print("    Esto NO se resuelve bajando el umbral ni ajustando pesos: la")
    print("    evidencia de manzana es externa y de mayor jerarquía que el")
    print("    puntaje. Si B3 los empareja, el método es demasiado permisivo.")
else:
    print("    Ninguna. B3 respeta los descartes de la manzana catastral.")

# -----------------------------------------------------------------------------
# Separación de los dos cambios: puntaje frente a alias de operador
#
# B1 comparaba operadores por similitud textual; B3 los resuelve por alias.
# Para saber cuánto de la diferencia viene de cada cosa, se reejecuta el
# emparejamiento con el ajuste de operador anulado: lo que quede es efecto
# puro del puntaje combinado.
# -----------------------------------------------------------------------------
SIN_OPERADOR, _, _ = emparejar(ocm, osm, bono=0.0, veto=0.0,
                               registrar_detalle=False)
s_sin_op = set(zip(SIN_OPERADOR.ocm_id, SIN_OPERADOR.osm_uid)) \
    if len(SIN_OPERADOR) else set()

print("\n" + "=" * 78)
print("DESCOMPOSICIÓN DE LA DIFERENCIA")
print("=" * 78)
print(f"  B1 (distancia recíproca):                  {len(s_b1)} pares")
print(f"  B3 sin ajuste de operador:                 {len(s_sin_op)} pares")
print(f"  B3 completo (con alias de operador):       {len(s_b3)} pares")
print(f"\n  Pares que mueve el puntaje combinado:      "
      f"{len(s_sin_op ^ s_b1)}")
print(f"  Pares que mueve el alias de operador:      {len(s_b3 ^ s_sin_op)}")
if s_b3 == s_sin_op:
    print("\n  El diccionario de alias no cambia NINGUNA asignación en estos")
    print("  datos. Debe reportarse así: es un criterio declarado que resulta")
    print("  inerte, no un criterio que esté decidiendo.")

# -----------------------------------------------------------------------------
# Tabla de contraste, para el informe
# -----------------------------------------------------------------------------
CONTRASTE = pd.DataFrame([
    {"metodo": "B1 · distancia recíproca", "pares": len(s_b1),
     "base": "emparejamiento por distancia mínima mutua"},
    {"metodo": "B2.1 · verificada", "pares": len(s_b21),
     "base": "B1 tras aplicar los veredictos de manzana catastral"},
    {"metodo": "B3 · sin ajuste de operador", "pares": len(s_sin_op),
     "base": "puntaje combinado, solo distancia y nombre"},
    {"metodo": "B3 · completo", "pares": len(s_b3),
     "base": "puntaje combinado con alias de operador"},
])
print("\n" + CONTRASTE.to_string(index=False))

MOVIMIENTOS_B3 = pd.DataFrame(
    [{"par": f"{o} <-> {s}", "ocm_id": o, "osm_uid": s,
      "en_B1": (o, s) in s_b1, "en_B2_1": (o, s) in s_b21,
      "en_B3": (o, s) in s_b3,
      "prohibido_por_B2": (o, s) in PARES_PROHIBIDOS}
     for o, s in sorted(s_b1 | s_b21 | s_b3 | PARES_PROHIBIDOS)]
)
print(f"\nPares que aparecen en al menos un método: {len(MOVIMIENTOS_B3)}")
print(MOVIMIENTOS_B3[["par", "en_B1", "en_B2_1", "en_B3",
                      "prohibido_por_B2"]].to_string(index=False))

LÍNEAS BASE
  B1  pares primarios válidos:        17   (+3 degradados, no eran pares)
  B2.1 pares tras verificación:       15
  B3  pares por puntaje combinado:    13

  Pares que B2 descartó con evidencia externa: 2
    470939 <-> node/13009702030
    475140 <-> node/13010000994

CONTRASTE 1 · B1 (distancia recíproca)  ->  B3 (puntaje combinado)
  En B1: 17   En B3: 13   Comunes: 13

--- Pares de B1 que B3 NO sostiene (4) ---
    470939 <-> node/13009702030
        EnelX Terminal Salitre  ||  Punto de carga Terminal de Transportes Salitre
        295.1 m · nombre 0.53 · categoría B1: C3 Desacuerdo posicional (sin verificar)
        B3 lo rechazó: puntaje insuficiente (puntaje 0.345)
    470940 <-> node/13010048854
        EnelX San Andresito  ||  Punto de carga Parque Los Ejidos
        82.5 m · nombre 0.12 · categoría B1: C2 Coincidencia probable
        B3 lo rechazó: puntaje insuficiente (puntaje 0.314)
    475122 <-> node/13009418551
        Terpel av Boyacá  ||  Punto de carga V

# Aporte real del operador

In [10]:
# =============================================================================
# Celda 18 — Aporte real del operador
# =============================================================================
# El operador figura en el método como un ajuste declarado: +0,10 si ambas
# fuentes resuelven al mismo alias, un veto si discrepan. Declararlo no es
# suficiente. Hay que medir si alguna vez cambia una asignación, porque un
# criterio que nunca decide nada no es un criterio: es un adorno del método
# que infla su apariencia de sofisticación y que un revisor va a pedir que se
# justifique con datos.
#
# Lo que ya sabemos y hace probable un resultado nulo:
#
#   - cerca de la mitad del inventario nacional de OpenChargeMap trae
#     '(Unknown Operator)', de modo que en buena parte de los pares el
#     operador simplemente no resuelve de un lado;
#   - en A4 se midió que el operador no discrimina después de normalizar
#     alias, porque las estaciones que comparten operador suelen compartir
#     también el nombre comercial, y el componente textual ya lo capta;
#   - el caso que motivó el veto —Terpel contra Voltex— resuelve al mismo
#     identificador en cuanto el diccionario existe, de modo que el veto no
#     puede activarse ahí.
#
# Un resultado nulo NO es un fracaso de la celda. Es el hallazgo, y hay que
# reportarlo con esas palabras en lugar de dejar el parámetro puesto como si
# estuviera haciendo algo.

# -----------------------------------------------------------------------------
# 1. Disponibilidad del campo, por fuente
#
# Tres estados, no dos: resuelto / sin dato / presente pero no declarado. Los
# dos últimos se corrigen de forma distinta y mezclarlos oculta cuál de los
# dos problemas es el dominante.
# -----------------------------------------------------------------------------
print("=" * 78)
print("1 · DISPONIBILIDAD DEL OPERADOR")
print("=" * 78)

DISPONIBILIDAD = []
for etiqueta, tabla in (("OCM", ocm), ("OSM", osm)):
    for ciudad in sorted(tabla.ciudad.unique()):
        sub = tabla[tabla.ciudad == ciudad]
        estados = [diagnosticar_operador(v)[1] for v in sub.operador_fuente]
        DISPONIBILIDAD.append({
            "fuente": etiqueta, "ciudad": ciudad, "registros": len(sub),
            "resuelto": sum(e == "resuelto" for e in estados),
            "sin_dato": sum(e.startswith("sin dato") for e in estados),
            "no_declarado": sum(e.startswith("no declarado") for e in estados),
        })
DISPONIBILIDAD = pd.DataFrame(DISPONIBILIDAD)
DISPONIBILIDAD["% resuelto"] = (
    DISPONIBILIDAD.resuelto / DISPONIBILIDAD.registros * 100).round(1)
print(DISPONIBILIDAD.to_string(index=False))
print("\n  'sin_dato' es limitación de la fuente; 'no_declarado' es tarea")
print("  pendiente nuestra en el diccionario de alias.")

# -----------------------------------------------------------------------------
# 2. Concordancia sobre los pares aceptados
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("2 · CONCORDANCIA SOBRE LOS PARES ACEPTADOS")
print("=" * 78)

if len(PARES_B3):
    _ambos = PARES_B3[PARES_B3.op_ocm.notna() & PARES_B3.op_osm.notna()]
    _acuerdo = _ambos[_ambos.op_ocm == _ambos.op_osm]
    _conflicto = _ambos[_ambos.op_ocm != _ambos.op_osm]

    print(f"  Pares aceptados:                          {len(PARES_B3)}")
    print(f"  Con operador resoluble en ambas fuentes:  {len(_ambos)}")
    print(f"    de acuerdo:                             {len(_acuerdo)}")
    print(f"    en conflicto:                           {len(_conflicto)}")
    print(f"  Con el operador no resoluble de algún lado: "
          f"{len(PARES_B3) - len(_ambos)}")

    if len(_conflicto):
        print("\n  Pares con operadores que discrepan:")
        print(_conflicto[["ocm_id", "nombre_ocm", "nombre_osm",
                          "op_ocm", "op_osm", "distancia_m", "puntaje"]]
              .round(3).to_string(index=False))
    else:
        print("\n  Ningún par aceptado tiene operadores en conflicto: el veto")
        print("  no tiene sobre qué actuar en este inventario.")

    print("\n  Reparto por vía de cálculo del puntaje:")
    for comp, n in PARES_B3.componentes.value_counts().items():
        print(f"    {n:>3}  {comp}")

# -----------------------------------------------------------------------------
# 3. Efecto del ajuste sobre el puntaje, sin reemparejar
#
# La función de puntaje devuelve 'puntaje_sin_ajuste'. Comparar las dos
# columnas sobre los mismos pares mide cuánto mueve el operador el VALOR del
# puntaje. Es distinto de si mueve la ASIGNACIÓN, que se mide en el punto 4.
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("3 · EFECTO DEL AJUSTE SOBRE EL VALOR DEL PUNTAJE")
print("=" * 78)

if len(PARES_B3):
    _mueve = PARES_B3[PARES_B3.ajuste_operador != 0]
    print(f"  Pares cuyo puntaje cambia por el ajuste: {len(_mueve)} "
          f"de {len(PARES_B3)}")

    # Los que SOLO pasan el umbral gracias al ajuste son el caso crítico: sin
    # el operador habrían quedado fuera.
    _dependientes = PARES_B3[(PARES_B3.puntaje >= UMBRAL_PUNTAJE)
                             & (PARES_B3.puntaje_sin_ajuste < UMBRAL_PUNTAJE)]
    print(f"  Pares que SOLO superan el umbral por el ajuste: "
          f"{len(_dependientes)}")
    if len(_dependientes):
        print(_dependientes[["ocm_id", "nombre_ocm", "nombre_osm",
                             "distancia_m", "puntaje_sin_ajuste", "puntaje"]]
              .round(3).to_string(index=False))
        print("\n  Estos son los pares que el operador sostiene por sí solo.")
        print("  Hay que poder defenderlos uno por uno.")
    else:
        print("  Ninguno: todos los pares aceptados superan el umbral sin")
        print("  necesidad del operador.")

# -----------------------------------------------------------------------------
# 4. Sensibilidad: ¿el ajuste cambia alguna ASIGNACIÓN?
#
# Se reejecuta el emparejamiento completo barriendo bono y veto. Los
# parámetros van como ARGUMENTOS: una versión anterior de esta celda mutaba
# la global con globals()[...] y la restauraba al final, de modo que una
# interrupción a mitad de celda dejaba el cuaderno con un parámetro distinto
# del declarado y sin aviso.
#
# EL VEREDICTO SE LEE POR RANGO, NO SOBRE LA TABLA ENTERA. Una versión
# anterior marcaba «activo» si alguna fila del barrido difería, sin distinguir
# si esa fila estaba dentro o fuera del rango plausible del parámetro. Con
# eso, un bono del doble del adoptado que mueve un solo par bastaba para
# declarar activo un criterio que en el valor adoptado no cambia nada. El
# texto contradecía la tabla impresa justo encima, y ese texto es el que se
# escribe en B3_parametros.json.
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("4 · SENSIBILIDAD DE LA ASIGNACIÓN")
print("=" * 78)

_ref = set(zip(PARES_B3.ocm_id, PARES_B3.osm_uid)) if len(PARES_B3) else set()

filas = []
for _bono in (0.00, 0.05, 0.10, 0.20):
    for _veto in (0.00, -0.10, -0.20, -0.30):
        P, _, _ = emparejar(ocm, osm, bono=_bono, veto=_veto,
                            registrar_detalle=False)
        s = set(zip(P.ocm_id, P.osm_uid)) if len(P) else set()
        filas.append({
            "bono": _bono, "veto": _veto, "pares": len(P),
            "difiere_del_adoptado": len(s ^ _ref),
            "es_el_adoptado": (_bono == BONO_OPERADOR and _veto == VETO_OPERADOR),
        })

SENSIBILIDAD_OPERADOR = pd.DataFrame(filas)
print(SENSIBILIDAD_OPERADOR.to_string(index=False))

_en_rango = SENSIBILIDAD_OPERADOR[
    SENSIBILIDAD_OPERADOR.bono <= BONO_OPERADOR]
_fuera = SENSIBILIDAD_OPERADOR[
    SENSIBILIDAD_OPERADOR.bono > BONO_OPERADOR]

BONO_ACTIVO_DESDE = None
print()
if _en_rango.difiere_del_adoptado.eq(0).all():
    VEREDICTO_OPERADOR = "inactivo en el valor adoptado"
    print("  VEREDICTO: el ajuste por operador es INACTIVO en el valor")
    print(f"  adoptado ({BONO_OPERADOR}) y en todo el rango probado por")
    print("  debajo de él: ninguna de esas combinaciones cambia una sola")
    print("  asignación, para ningún valor del veto.")
    if len(PARES_B3):
        print(f"\n  La causa es medible y está en el punto 1: el campo no")
        print(f"  resuelve de ambos lados en {len(PARES_B3) - len(_ambos)} de "
              f"los {len(PARES_B3)} pares aceptados,")
        print(f"  y donde resuelve, los {len(_acuerdo)} casos están de acuerdo. "
              "El veto no")
        print("  tiene sobre qué actuar y el bono no tiene a quién desempatar.")
    if len(_fuera) and _fuera.difiere_del_adoptado.gt(0).any():
        BONO_ACTIVO_DESDE = float(
            _fuera[_fuera.difiere_del_adoptado > 0].bono.min())
        print(f"\n  El parámetro solo empieza a decidir a partir de un bono de")
        print(f"  {BONO_ACTIVO_DESDE}, es decir "
              f"{BONO_ACTIVO_DESDE / BONO_OPERADOR:.0f} veces el valor "
              "adoptado. Ese punto se")
        print("  reporta como el límite a partir del cual el criterio dejaría")
        print("  de ser inerte, NO como evidencia de que hoy actúe.")
    print("\n  Debe declararse como criterio previsto que no interviene, y no")
    print("  presentarse como si estuviera decidiendo.")
else:
    VEREDICTO_OPERADOR = "activo"
    _filas_activas = _en_rango[_en_rango.difiere_del_adoptado > 0]
    print("  VEREDICTO: el ajuste por operador SÍ cambia asignaciones dentro")
    print("  del rango adoptado. El parámetro es una decisión con")
    print("  consecuencias y hay que justificarla más allá de declararla.")
    print(f"\n  Combinaciones dentro del rango que producen un resultado")
    print(f"  distinto del adoptado: {len(_filas_activas)}")
    print(_filas_activas.to_string(index=False))

# -----------------------------------------------------------------------------
# 5. El caso que motivó el veto
#
# En B1 quedó señalado para B3 el par «Terpel Las Vegas» contra «Voltex Las
# Vegas», a 30,7 m, con 0,75 de similitud de cadena, descrito como dos
# estaciones de operadores distintos que el parecido del nombre del lugar
# podía fusionar indebidamente. Conviene mostrar qué pasa hoy con él, porque
# el informe cita el caso y la lectura cambió.
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("5 · EL CASO QUE MOTIVÓ EL VETO")
print("=" * 78)

_lasvegas = pd.DataFrame()
if len(CANDIDATOS):
    _m = (CANDIDATOS.nombre_ocm.fillna("").str.contains("vegas", case=False)
          | CANDIDATOS.nombre_osm.fillna("").str.contains("vegas", case=False))
    _lasvegas = CANDIDATOS[_m]

if len(_lasvegas):
    print(_lasvegas[["ocm_id", "nombre_ocm", "nombre_osm", "op_ocm", "op_osm",
                     "distancia_m", "s_nombre", "puntaje", "componentes"]]
          .round(3).to_string(index=False))
    print("\n  Terpel y Voltex resuelven al mismo identificador: Voltex es la")
    print("  marca de electrolineras de Terpel. El diagnóstico original de B1")
    print("  los leyó como empresas distintas, y esa lectura hay que")
    print("  corregirla en el informe. El veto, pensado para este caso, no")
    print("  puede activarse aquí.")
else:
    print("  No se encontraron candidatos con ese nombre en el radio de")
    print("  búsqueda. El caso debe rastrearse por identificador antes de")
    print("  citarlo en el informe.")

print("\n" + "=" * 78)
print(f"VEREDICTO SOBRE EL OPERADOR: {VEREDICTO_OPERADOR.upper()}")
print("=" * 78)

1 · DISPONIBILIDAD DEL OPERADOR
fuente   ciudad  registros  resuelto  sin_dato  no_declarado  % resuelto
   OCM   Bogota         27        19         8             0        70.4
   OCM Medellin         10         5         4             1        50.0
   OSM   Bogota         41        16        22             3        39.0
   OSM Medellin         11         4         6             1        36.4

  'sin_dato' es limitación de la fuente; 'no_declarado' es tarea
  pendiente nuestra en el diccionario de alias.

2 · CONCORDANCIA SOBRE LOS PARES ACEPTADOS
  Pares aceptados:                          13
  Con operador resoluble en ambas fuentes:  6
    de acuerdo:                             6
    en conflicto:                           0
  Con el operador no resoluble de algún lado: 7

  Ningún par aceptado tiene operadores en conflicto: el veto
  no tiene sobre qué actuar en este inventario.

  Reparto por vía de cálculo del puntaje:
      6  distancia+nombre+operador_igual
      6  distancia

# Elección del umbral: Sensibilidad y criterio declarado

In [11]:
# Calibrar un umbral exige un conjunto de casos etiquetados con ambos signos:
# parejas que se sabe que son la misma estación y parejas que se sabe que no.
# Con eso se mide, para cada corte, cuántos falsos positivos y cuántos falsos
# negativos produce, y se elige donde el costo total es mínimo.
#
# Lo que tenemos es otra cosa. B2 resolvió DOS de los seis casos declarados
# —C3-2 y C3-3— y los dos son NEGATIVOS: manzanas catastrales distintas, luego
# estaciones distintas. Los otros cuatro quedaron no concluyentes y, por regla
# explícita de B2.1, un veredicto None nunca se convierte en False.
#
# Con dos etiquetas del mismo signo:
#   - se puede comprobar que ningún umbral razonable reconstituya lo que la
#     manzana descartó. Eso es una PRUEBA DE ACEPTACIÓN, y está en su celda.
#   - NO se puede medir el falso negativo, porque no hay un solo caso positivo
#     verificado externamente contra el cual medirlo.
#
# El umbral se elige entonces con dos insumos honestos:
#
#   1. UNA TABLA DE SENSIBILIDAD que muestra qué pares entran y salen en cada
#      escalón, de modo que la elección se lee con sus consecuencias a la
#      vista y no como un número dado.
#
#   2. UN CRITERIO DE COSTO ASIMÉTRICO, declarado antes de mirar los números:
#      un falso positivo —dos estaciones distintas fusionadas en una— es más
#      grave que un falso negativo —una estación contada como dos registros—,
#      porque el inventario está destinado a depósito con identificador
#      persistente. Un registro de más se corrige después y queda visible; una
#      fusión indebida borra una estación del inventario y nadie la echa de
#      menos. Ante empate en la evidencia, se elige el umbral más alto.

CRITERIO_UMBRAL = (
    "costo asimétrico: ante evidencia equivalente se prefiere el umbral más "
    "alto, porque un falso positivo elimina una estación del inventario y un "
    "falso negativo solo la duplica"
)

# -----------------------------------------------------------------------------
# Veredictos externos disponibles
# -----------------------------------------------------------------------------
VEREDICTOS_B2, FICHA_B2 = constantes.cargar_veredictos_b2()

_adoptados = {c: v for c, v in VEREDICTOS_B2.items() if v is not None}
_sin_concluir = sorted(c for c, v in VEREDICTOS_B2.items() if v is None)

print("=" * 78)
print("VEREDICTOS EXTERNOS DISPONIBLES")
print("=" * 78)
print(f"  Casos declarados por B1:        {len(VEREDICTOS_B2)}")
print(f"  Con conclusión adoptada por B2: {len(_adoptados)}  "
      f"{sorted(_adoptados)}")
print(f"  Sin conclusión:                 {len(_sin_concluir)}  {_sin_concluir}")
print(f"  Positivos (misma estación):     "
      f"{sum(1 for v in _adoptados.values() if v)}")
print(f"  Negativos (estaciones distintas): "
      f"{sum(1 for v in _adoptados.values() if not v)}")
print("\n  Sin casos positivos verificados no hay forma de medir el falso")
print("  negativo. La tabla de abajo es sensibilidad, no calibración.")

# -----------------------------------------------------------------------------
# Tabla de sensibilidad
#
# Se recorre CANDIDATOS en vez de reevaluar el producto cartesiano: todos los
# umbrales miden sobre exactamente el mismo conjunto de candidatos, que es la
# condición para que la comparación entre escalones signifique algo.
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("SENSIBILIDAD AL UMBRAL")
print("=" * 78)

_conjuntos, filas = {}, []
for u in UMBRALES_EVALUADOS:
    P, _, _ = emparejar(ocm, osm, umbral=u, registrar_detalle=False)
    s = set(zip(P.ocm_id, P.osm_uid)) if len(P) else set()
    _conjuntos[u] = s

    _prohibidos = len(s & PARES_PROHIBIDOS)
    _recupera_b1 = len(s & s_b1)
    _recupera_b21 = len(s & s_b21)

    filas.append({
        "umbral": u,
        "pares": len(s),
        "de_B1_recuperados": f"{_recupera_b1}/{len(s_b1)}",
        "de_B2.1_recuperados": f"{_recupera_b21}/{len(s_b21)}",
        "contradice_a_B2": _prohibidos,
        "adoptado": "<==" if abs(u - UMBRAL_PUNTAJE) < 1e-9 else "",
    })

SENSIBILIDAD_UMBRAL = pd.DataFrame(filas)
print(SENSIBILIDAD_UMBRAL.to_string(index=False))

# -----------------------------------------------------------------------------
# Qué par entra y sale en cada escalón
#
# El conteo agregado no basta: dos umbrales pueden dar el mismo número de
# pares y no ser el mismo conjunto. Lo que decide es CUÁLES cambian.
# -----------------------------------------------------------------------------
print("\n" + "-" * 78)
print("MOVIMIENTOS ENTRE ESCALONES")
print("-" * 78)

_orden = sorted(UMBRALES_EVALUADOS)
for anterior, siguiente in zip(_orden, _orden[1:]):
    salen = _conjuntos[anterior] - _conjuntos[siguiente]
    entran = _conjuntos[siguiente] - _conjuntos[anterior]
    if not salen and not entran:
        print(f"  {anterior:.2f} -> {siguiente:.2f}   sin cambios")
        continue
    print(f"  {anterior:.2f} -> {siguiente:.2f}")
    for oid, ouid in sorted(salen):
        f = PARES_B3[(PARES_B3.ocm_id == oid) & (PARES_B3.osm_uid == ouid)]
        det = ""
        if len(f):
            r = f.iloc[0]
            det = (f"  [{r.distancia_m:.0f} m · puntaje {r.puntaje:.3f}]  "
                   f"{r.nombre_ocm} || {r.nombre_osm}")
        marca = "  ** prohibido por B2 **" if (oid, ouid) in PARES_PROHIBIDOS else ""
        print(f"      SALE   {oid} <-> {ouid}{det}{marca}")
    for oid, ouid in sorted(entran):
        print(f"      ENTRA  {oid} <-> {ouid}")

# -----------------------------------------------------------------------------
# Umbrales admisibles según la evidencia externa
#
# Único filtro duro que los veredictos permiten aplicar: un umbral que
# reconstituya un par que la manzana catastral descartó es inadmisible. No es
# cuestión de ajuste fino; la evidencia de manzana es externa y de mayor
# jerarquía que un puntaje calculado sobre las dos fuentes que ya estaban en
# desacuerdo.
# -----------------------------------------------------------------------------
print("\n" + "-" * 78)
print("ADMISIBILIDAD SEGÚN LA VERIFICACIÓN EXTERNA")
print("-" * 78)

ADMISIBLES = [u for u in UMBRALES_EVALUADOS
              if not (_conjuntos[u] & PARES_PROHIBIDOS)]
INADMISIBLES = [u for u in UMBRALES_EVALUADOS if u not in ADMISIBLES]

print(f"  Admisibles:   {ADMISIBLES}")
print(f"  Inadmisibles: {INADMISIBLES if INADMISIBLES else 'ninguno'}")

if not INADMISIBLES:
    print("\n  Ningún umbral del rango probado contradice a B2. La evidencia")
    print("  externa no discrimina entre ellos: no permite elegir, solo")
    print("  permite descartar, y no hay nada que descartar. La elección")
    print("  recae íntegramente en el criterio de costo declarado arriba.")

assert UMBRAL_PUNTAJE in ADMISIBLES, (
    f"El umbral adoptado ({UMBRAL_PUNTAJE}) reconstituye un par que B2 "
    "descartó con la manzana catastral. No se corrige moviendo pesos: el "
    "método es demasiado permisivo."
)

# -----------------------------------------------------------------------------
# Estabilidad del resultado
#
# Si el conjunto de pares no cambia en un tramo del barrido, el resultado es
# robusto a la elección del umbral dentro de ese tramo, y conviene reportarlo
# así: es más informativo que el número elegido.
# -----------------------------------------------------------------------------
print("\n" + "-" * 78)
print("ESTABILIDAD")
print("-" * 78)

_tramos, _actual = [], [_orden[0]]
for anterior, siguiente in zip(_orden, _orden[1:]):
    if _conjuntos[anterior] == _conjuntos[siguiente]:
        _actual.append(siguiente)
    else:
        _tramos.append(_actual)
        _actual = [siguiente]
_tramos.append(_actual)

for tramo in _tramos:
    marca = " <== contiene el umbral adoptado" if UMBRAL_PUNTAJE in tramo else ""
    print(f"  [{tramo[0]:.2f} – {tramo[-1]:.2f}]  "
          f"{len(_conjuntos[tramo[0]])} pares{marca}")

_tramo_adoptado = next(t for t in _tramos if UMBRAL_PUNTAJE in t)
print(f"\n  El umbral adoptado ({UMBRAL_PUNTAJE}) vive en un tramo de "
      f"{len(_tramo_adoptado)} valor(es)")
print(f"  probados, de {_tramo_adoptado[0]:.2f} a {_tramo_adoptado[-1]:.2f}, "
      "todos con el mismo resultado.")
if len(_tramo_adoptado) == 1:
    print("  ATENCIÓN: el resultado cambia al mover el umbral un solo escalón.")
    print("  Es frágil, y la fragilidad debe reportarse junto con el número.")

# -----------------------------------------------------------------------------
DECISION_UMBRAL = {
    "umbral_adoptado": UMBRAL_PUNTAJE,
    "umbrales_evaluados": UMBRALES_EVALUADOS,
    "criterio": CRITERIO_UMBRAL,
    "es_calibracion": False,
    "motivo_no_es_calibracion": (
        "B2 adoptó 2 veredictos de 6, ambos negativos; sin casos positivos "
        "verificados no se puede medir el falso negativo. La evidencia "
        "externa permite descartar umbrales, no elegir entre los admisibles."
    ),
    "veredictos_adoptados": {c: bool(v) for c, v in _adoptados.items()},
    "veredictos_sin_concluir": _sin_concluir,
    "umbrales_admisibles": ADMISIBLES,
    "umbrales_inadmisibles": INADMISIBLES,
    "tramo_de_estabilidad": [float(_tramo_adoptado[0]), float(_tramo_adoptado[-1])],
    "pares_resultantes": len(_conjuntos[UMBRAL_PUNTAJE]),
}

print("\n" + "=" * 78)
print(f"UMBRAL ADOPTADO: {UMBRAL_PUNTAJE}  ({len(_conjuntos[UMBRAL_PUNTAJE])} pares)")
print("=" * 78)
print(f"  Criterio: {CRITERIO_UMBRAL}")

VEREDICTOS EXTERNOS DISPONIBLES
  Casos declarados por B1:        6
  Con conclusión adoptada por B2: 2  ['C3-2', 'C3-3']
  Sin conclusión:                 4  ['C3-1', 'N1-1', 'N1-2', 'N1-3']
  Positivos (misma estación):     0
  Negativos (estaciones distintas): 2

  Sin casos positivos verificados no hay forma de medir el falso
  negativo. La tabla de abajo es sensibilidad, no calibración.

SENSIBILIDAD AL UMBRAL
 umbral  pares de_B1_recuperados de_B2.1_recuperados  contradice_a_B2 adoptado
   0.40     15             14/17               14/15                0         
   0.45     15             14/17               14/15                0         
   0.50     13             13/17               13/15                0      <==
   0.55     12             12/17               12/15                0         
   0.60     11             11/17               11/15                0         
   0.65     10             10/17               10/15                0         
   0.70      8              

# Pruebas y guardado

In [12]:
PRUEBAS = []


def prueba(nombre, condicion, detalle=""):
    PRUEBAS.append({"prueba": nombre, "pasa": bool(condicion), "detalle": detalle})


# -----------------------------------------------------------------------------
# 1. Distancia verificada sobre el elipsoide
#
# Las distancias del cuaderno vienen de proyectar a EPSG:9377 y medir en el
# plano. Aquí se recalculan como geodésicas sobre GRS80 —el mismo elipsoide,
# otro camino— recuperando las coordenadas POR IDENTIFICADOR desde las
# fuentes, no desde la tabla de pares: así la prueba no hereda el eventual
# error de unión que busca detectar.
#
# Los dos métodos no coinciden exactamente y no deben hacerlo. EPSG:9377 es
# una Transverse Mercator con meridiano central en -73° y k0 = 0,9992, de modo
# que la escala está por debajo de 1 cerca del meridiano y por encima pasados
# unos 255 km. Bogotá está a ~119 km y SUBESTIMA (~-640 ppm); Medellín a
# ~284 km y SOBREESTIMA (~+209 ppm). El signo es distinto en cada ciudad: B1
# afirma «subestima» para las dos, y esa redacción hay que corregirla allá.
# Sobre el umbral de 25 m son ~1,5 cm, sin efecto sobre ninguna clasificación.
# -----------------------------------------------------------------------------
_geod = Geod(ellps="GRS80")

VERIFICACION_DISTANCIAS = pd.DataFrame()
if len(PARES_B3):
    c = PARES_B3[["ciudad", "ocm_id", "osm_uid", "distancia_m"]].copy()
    c = c.merge(ocm[["id", "lat", "lon"]].rename(
        columns={"id": "ocm_id", "lat": "lat_ocm", "lon": "lon_ocm"}),
        on="ocm_id", how="left")
    c = c.merge(osm[["id", "lat", "lon"]].rename(
        columns={"id": "osm_uid", "lat": "lat_osm", "lon": "lon_osm"}),
        on="osm_uid", how="left")

    if c[["lat_ocm", "lat_osm"]].isna().any().any():
        raise AssertionError(
            "Hay pares cuyas coordenadas no se pudieron recuperar por "
            "identificador: el id reportado no existe en la fuente."
        )

    _, _, geodesica = _geod.inv(c.lon_ocm.values, c.lat_ocm.values,
                                c.lon_osm.values, c.lat_osm.values)
    dif = np.abs(geodesica - c.distancia_m.values)
    tol = np.maximum(0.05, c.distancia_m.values * 0.001)

    VERIFICACION_DISTANCIAS = pd.DataFrame({
        "ciudad": c.ciudad, "ocm_id": c.ocm_id, "osm_uid": c.osm_uid,
        "reportada_m": c.distancia_m.round(2),
        "geodesica_m": np.round(geodesica, 2),
        "diferencia_m": dif.round(3),
        "tolerancia_m": tol.round(3),
        "ppm": np.where(c.distancia_m.values > 0,
                        (geodesica - c.distancia_m.values)
                        / c.distancia_m.values * 1e6, np.nan).round(0),
        "pasa": dif <= tol,
    })
    prueba("la distancia coincide con la geodésica",
           VERIFICACION_DISTANCIAS.pasa.all(),
           f"{int(VERIFICACION_DISTANCIAS.pasa.sum())}/"
           f"{len(VERIFICACION_DISTANCIAS)} pares")

# -----------------------------------------------------------------------------
# 2. Reducción a B1
#
# Con peso_nombre = 0 y sin ajuste de operador, el puntaje queda monótono
# decreciente en la distancia, de modo que el emparejamiento recíproco por
# puntaje TIENE que reproducir los pares mutuos por distancia de B1. Si no lo
# hace, el defecto está en emparejar() y no en el criterio combinado.
#
# Es la prueba más valiosa del cuaderno: ata el método nuevo al anterior en el
# único punto donde ambos deben coincidir por construcción.
# -----------------------------------------------------------------------------
REDUCCION = emparejar_mutuo(ocm, osm, umbral=0.0,
                            peso_distancia=1.0, peso_nombre=0.0,
                            bono=0.0, veto=0.0)
s_red = set(zip(REDUCCION.ocm_id, REDUCCION.osm_uid)) if len(REDUCCION) else set()
s_b1_todos = set(zip(B1P.ocm_id, B1P.osm_uid))

prueba("reducido a distancia pura, reproduce los pares mutuos de B1",
       s_red == s_b1_todos,
       f"{len(s_red & s_b1_todos)} comunes de {len(s_b1_todos)}; "
       f"solo B1 {sorted(s_b1_todos - s_red)}; solo reducción {sorted(s_red - s_b1_todos)}")

# -----------------------------------------------------------------------------
# 3. No contradicción con la verificación externa
# -----------------------------------------------------------------------------
prueba("ningún par contradice un descarte de la manzana catastral",
       len(s_b3 & PARES_PROHIBIDOS) == 0,
       f"{len(PARES_PROHIBIDOS)} pares prohibidos evaluados")

# -----------------------------------------------------------------------------
# 4. Trazabilidad de cada par aceptado
#
# Todo par aceptado tiene que existir en CANDIDATOS con el mismo puntaje. Un
# desalineamiento de índices entre la matriz de distancias y la unión de
# atributos —el error original del Bloque A— haría que la fila reportada no
# fuera la que produjo el cálculo.
# -----------------------------------------------------------------------------
if len(PARES_B3) and len(CANDIDATOS):
    clave = ["ocm_id", "osm_uid"]
    cruce = PARES_B3[clave + ["puntaje", "distancia_m"]].merge(
        CANDIDATOS[clave + ["puntaje", "distancia_m"]],
        on=clave, how="left", suffixes=("_par", "_cand"), indicator=True)
    prueba("cada par aceptado existe en los candidatos, con el mismo puntaje",
           (cruce._merge == "both").all()
           and np.allclose(cruce.puntaje_par, cruce.puntaje_cand)
           and np.allclose(cruce.distancia_m_par, cruce.distancia_m_cand),
           f"{len(cruce)} pares")

# -----------------------------------------------------------------------------
# 5. Los identificadores existen en las fuentes deduplicadas
# -----------------------------------------------------------------------------
if len(PARES_B3):
    prueba("todo ocm_id emparejado existe en el inventario deduplicado",
           set(PARES_B3.ocm_id) <= set(ocm.id.astype(str)))
    prueba("todo osm_uid emparejado existe en el inventario deduplicado",
           set(PARES_B3.osm_uid) <= set(osm.id.astype(str)))

# -----------------------------------------------------------------------------
# 6. Determinismo frente al orden de entrada
#
# El resultado no puede depender de cómo vengan ordenadas las filas. Se
# reordena la entrada al revés y se vuelve a emparejar: si el desempate por
# identificador se quitara, esta prueba lo detectaría.
# -----------------------------------------------------------------------------
P_rev, _, _ = emparejar(ocm.iloc[::-1].reset_index(drop=True),
                        osm.iloc[::-1].reset_index(drop=True),
                        registrar_detalle=False)
s_rev = set(zip(P_rev.ocm_id, P_rev.osm_uid)) if len(P_rev) else set()
prueba("el resultado no depende del orden de las filas de entrada",
       s_rev == s_b3, f"{len(s_rev ^ s_b3)} diferencias")

# -----------------------------------------------------------------------------
# 7. Codicioso frente a recíproco
#
# Informativa, no bloqueante: una diferencia significa que hubo registros
# disputados, lo cual es legítimo, pero debe quedar reportada.
# -----------------------------------------------------------------------------
prueba("la asignación codiciosa coincide con la recíproca",
       _s_cod == _s_mut,
       f"{len(_s_cod ^ _s_mut)} diferencias; ver celda de emparejamiento")

# -----------------------------------------------------------------------------
# 8. Los insumos no cambiaron mientras corría el cuaderno
# -----------------------------------------------------------------------------
for etiqueta, ruta in (("B1_pares_primarios.csv", ARCH_PARES_B1),
                       ("B2_1_clasificacion_verificada.csv", ARCH_VERIFICADA)):
    prueba(f"{etiqueta} no fue modificado durante la ejecución",
           constantes.sha256_archivo(ruta) == HASHES_INSUMOS[etiqueta])

# -----------------------------------------------------------------------------
RESUMEN_PRUEBAS = pd.DataFrame(PRUEBAS)
print("=" * 78)
print("PRUEBAS DE ACEPTACIÓN")
print("=" * 78)
for p in PRUEBAS:
    print(f"  [{'OK   ' if p['pasa'] else 'FALLA'}] {p['prueba']}")
    if p["detalle"]:
        print(f"           {p['detalle']}")

TODAS_PASAN = bool(RESUMEN_PRUEBAS.pasa.all())
print(f"\n  {int(RESUMEN_PRUEBAS.pasa.sum())} de {len(RESUMEN_PRUEBAS)} pasan")

if len(VERIFICACION_DISTANCIAS):
    print("\n  Factor de escala de EPSG:9377 medido por ciudad (ppm):")
    print(VERIFICACION_DISTANCIAS.groupby("ciudad").ppm
          .agg(["mean", "min", "max", "count"]).round(0).to_string())
    print("  Signo positivo = la proyección subestima; negativo = sobreestima")

# =============================================================================
# GUARDADO
# =============================================================================
# Sin marca de tiempo de ejecución en ningún archivo: un run-time cambia el
# hash en cada corrida y lo vuelve inútil como huella. Solo se registran
# fechas que son hechos sobre el RESULTADO, no sobre la corrida.

ALIAS_TABLA = pd.DataFrame(
    [{"alias": k, "identificador": v,
      "justificacion": JUSTIFICACION_ALIAS.get(v, "")}
     for k, v in sorted(ALIAS_OPERADOR.items())])

SALIDAS = {
    # Resultado principal
    "B3_pares_puntaje_combinado.csv": (PARES_B3, False),
    "B3_candidatos_evaluados.csv": (CANDIDATOS, False),
    "B3_candidatos_rechazados.csv": (RECHAZADOS, False),

    # Lo que justifica el cambio de método
    "B3_reasignaciones.csv": (REASIGNACIONES, False),
    "B3_contraste_metodos.csv": (CONTRASTE, False),
    "B3_movimientos_por_par.csv": (MOVIMIENTOS_B3, False),

    # Parámetros y su sustento
    "B3_sensibilidad_umbral.csv": (SENSIBILIDAD_UMBRAL, False),
    "B3_sensibilidad_operador.csv": (SENSIBILIDAD_OPERADOR, False),
    "B3_diccionario_alias_operador.csv": (ALIAS_TABLA, False),
    "B3_cobertura_operador.csv": (DISPONIBILIDAD, False),

    # Deduplicación interna
    "B3_fusiones_internas.csv": (FUSIONES, False),
    "B3_duplicados_no_evaluables.csv": (NO_EVALUABLES, False),

    # Verificación
    "B3_pruebas_aceptacion.csv": (RESUMEN_PRUEBAS, False),
    "B3_verificacion_distancias.csv": (VERIFICACION_DISTANCIAS, False),
}

print("\n" + "=" * 78)
print("GUARDADO")
print("=" * 78)
for nombre, (tabla, con_indice) in SALIDAS.items():
    tabla.to_csv(RUTA_OUT / nombre, index=con_indice, encoding="utf-8-sig")
    print(f"  {nombre:42s} {len(tabla):>5d} filas")

# --- Parámetros y decisiones -------------------------------------------------
PARAMETROS = {
    "bloque": "B3",
    "proposito": ("reemplazar el emparejamiento por distancia de B1 por un "
                  "puntaje combinado y contrastar ambos resultados"),
    "fecha_corte_inventario": constantes.FECHA_CORTE_LOCAL,

    "escala_distancia_m": ESCALA_DISTANCIA_M,
    "origen_escala": {
        "medida": "lado largo mediano de manzana en el entorno de las estaciones",
        "fuente": ESCALA_FUENTE,
        "ciudad_medida": ESCALA_CIUDAD_MEDIDA,
        "limitacion": ("la medida es de Bogotá; no se buscó una capa de "
                       "manzanas comparable para Medellín, de modo que la "
                       "escala se transfiere allá sin verificar"),
    },

    "peso_distancia": PESO_DISTANCIA,
    "peso_nombre": PESO_NOMBRE,
    "bono_operador": BONO_OPERADOR,
    "veto_operador": VETO_OPERADOR,
    "veredicto_sobre_el_operador": VEREDICTO_OPERADOR,
    "bono_operador_activo_desde": BONO_ACTIVO_DESDE,
    "radio_busqueda_m": RADIO_BUSQUEDA_MAX,

    "decision_umbral": DECISION_UMBRAL,

    "resultados": {
        "pares_B1_validos": len(s_b1),
        "pares_B2_1_verificados": len(s_b21),
        "pares_B3": len(s_b3),
        "reasignaciones_respecto_de_la_distancia": len(REASIGNACIONES),
        "contradicciones_con_B2": len(s_b3 & PARES_PROHIBIDOS),
    },

    "limitaciones_declaradas": [
        "la escala espacial se midió solo en Bogotá",
        ("no hay casos positivos verificados externamente, de modo que el "
         "umbral no está calibrado sino elegido por criterio de costo"),
        ("el contraste contra B1 mezcla dos cambios —puntaje combinado y "
         "alias de operador—; se separan en B3_contraste_metodos.csv"),
    ],

    "atribucion": [constantes.ATRIBUCION_ODBL, constantes.ATRIBUCION_OCM],
    "pruebas_pasan": TODAS_PASAN,
}

with open(RUTA_OUT / "B3_parametros.json", "w", encoding="utf-8") as fh:
    json.dump(PARAMETROS, fh, ensure_ascii=False, indent=2)

# --- Manifiesto de insumos ---------------------------------------------------
# Mismo formato plano de B2 y B2.1, para que la cadena se recorra hacia atrás
# archivo por archivo hasta los crudos congelados de A2 y A3. Las entradas
# 'crudo::' se heredan sin recalcular: B2 ya las validó.
MANIFIESTO = dict(HASHES_INSUMOS)
MANIFIESTO.update(CRUDOS_HEREDADOS)
with open(RUTA_OUT / "B3_manifiesto_insumos.json", "w", encoding="utf-8") as fh:
    json.dump(MANIFIESTO, fh, ensure_ascii=False, indent=2, sort_keys=True)

# --- Relectura de ida y vuelta ----------------------------------------------
# Lo escrito tiene que releerse igual. Se compara con .compare().empty y no
# con .equals(), que compara también los dtypes: en B2.1 eso produjo dos
# fallos falsos tras una concatenación y tras una relectura.
if len(PARES_B3):
    _releido = pd.read_csv(RUTA_OUT / "B3_pares_puntaje_combinado.csv",
                           encoding="utf-8-sig",
                           dtype={"ocm_id": "string", "osm_uid": "string"})
    _orig = PARES_B3.reset_index(drop=True)
    _iguales = (set(zip(_releido.ocm_id, _releido.osm_uid))
                == set(zip(_orig.ocm_id, _orig.osm_uid))
                and np.allclose(_releido.puntaje, _orig.puntaje))
    print(f"\n  Relectura de ida y vuelta: {'OK' if _iguales else 'FALLA'}")

# --- Huella de las salidas ---------------------------------------------------
HUELLAS = {n: constantes.sha256_archivo(RUTA_OUT / n) for n in SALIDAS}
HUELLAS["B3_parametros.json"] = constantes.sha256_archivo(
    RUTA_OUT / "B3_parametros.json")
with open(RUTA_OUT / "B3_huellas_salidas.json", "w", encoding="utf-8") as fh:
    json.dump(HUELLAS, fh, ensure_ascii=False, indent=2, sort_keys=True)

print(f"\n  Archivos en: {RUTA_OUT}")
print(f"  Pruebas de aceptación: "
      f"{'TODAS PASAN' if TODAS_PASAN else 'HAY FALLAS — revisar antes de reportar'}")
print("\n  Las salidas de B1, B2 y B2.1 quedan intactas.")

PRUEBAS DE ACEPTACIÓN
  [OK   ] la distancia coincide con la geodésica
           13/13 pares
  [OK   ] reducido a distancia pura, reproduce los pares mutuos de B1
           20 comunes de 20; solo B1 []; solo reducción []
  [OK   ] ningún par contradice un descarte de la manzana catastral
           2 pares prohibidos evaluados
  [OK   ] cada par aceptado existe en los candidatos, con el mismo puntaje
           13 pares
  [OK   ] todo ocm_id emparejado existe en el inventario deduplicado
  [OK   ] todo osm_uid emparejado existe en el inventario deduplicado
  [OK   ] el resultado no depende del orden de las filas de entrada
           0 diferencias
  [OK   ] la asignación codiciosa coincide con la recíproca
           0 diferencias; ver celda de emparejamiento
  [OK   ] B1_pares_primarios.csv no fue modificado durante la ejecución
  [OK   ] B2_1_clasificacion_verificada.csv no fue modificado durante la ejecución

  10 de 10 pasan

  Factor de escala de EPSG:9377 medido por ciudad (ppm

# Decisión pendiente sobre el umbral

## Umbral adoptado: 0,50

Ningún umbral del rango contradice los descartes de B2, de modo que la
evidencia externa no permite elegir entre ellos. La decisión se toma sobre los
dos pares que separan 0,50 de 0,60:

| par | distancia | sim. nombre | puntaje |
|---|---|---|---|
| Compensar / Punto de carga Compensar Avenida 68 | 43,0 m | 0,41 | 0,557 |
| EnelX Nicolás de Federmán / Punto de carga Clínica Nicolás de Federmán | 139,0 m | 0,66 | 0,516 |

Compensar es el caso decisivo: a 43 metros, con el nombre de una fuente
contenido literalmente en el de la otra. No cae por evidencia en contra sino
por una limitación conocida de la medida textual, que penaliza la contención
—Jaccard 0,17, secuencia 0,41— aunque sea el caso más claro de identidad.
Rechazarlo sería un falso negativo indefendible.

Nicolás de Federmán se acepta de forma coherente con B2, que lo dejó como no
concluyente: la manzana catastral no discriminó, de modo que no hay evidencia
externa en contra.

El criterio de costo asimétrico se declaró como desempate ante evidencia
equivalente. Aquí la evidencia no es equivalente, de modo que no aplica.

LIMITACIÓN: 0,50 vive en un tramo de estabilidad de un solo valor probado. El
resultado cambia al mover el umbral un escalón en cualquier dirección, y esa
fragilidad debe reportarse junto con el número.

La reasignación de San Andresito solo aparece en el tramo 0,40–0,45 y no se
recupera a 0,50. `REASIGNACIONES = 0` se mantiene.